In [1]:
# %% Cell -1: Control K3D plotting in this notebook
RUN_K3D_FOR_INTERP_DEBUG = True 


In [2]:
# %% Cell 0: Setup and Imports
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import h5py
from tqdm.notebook import tqdm
from pyquaternion import Quaternion
import logging
from typing import Dict, List, Optional, Tuple, Any
import k3d
import json

# --- Add Project Root to sys.path ---
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(f"Added project root to sys.path: {PROJECT_ROOT}")

# --- Import Custom Modules ---
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import Box as NuScenesDataClassesBox

from src.config_loader import MDetectorConfigAccessor
from src.core.m_detector.base import MDetector
from src.core.depth_image import DepthImage
from src.core.constants import OcclusionResult, POINT_LABEL_DTYPE
from src.core.debug_collector import PointDebugCollector
from src.data_utils.nuscenes_helper import get_scene_sweep_data_sequence, get_lidar_sweep_data
from src.utils.transformations import transform_points_numpy
from src.utils.validation_utils import get_gt_dynamic_points_for_sweep
from src.utils.notebook_helpers import create_k3d_plot_with_points
# Import the new interpolation utility
from src.core.m_detector.interpolation_utils import interpolate_surface_depth_at_angle # Ensure this path is correct

print("Cell 0: Imports and Paths - OK")


Added project root to sys.path: /home/drugge/Unsupervised-Moving-Point-Detection/m_detector_python
Cell 0: Imports and Paths - OK


In [3]:

# %% Cell 1: Logging, Configuration, NuScenes Init, Paths, Scene/Sweep Selection
# This cell is largely the same as your Cell 1 in the visualization notebook.
# Ensure 'm_detector_config.yaml' includes the new 'map_consistency.interpolation_*' parameters.

# --- Configure Logging ---
# For this notebook, set MCC and Interpolation loggers to DEBUG
logging.basicConfig(level=logging.INFO) # General level
logging.getLogger('src.core.m_detector.base').setLevel(logging.INFO)
logging.getLogger('src.core.m_detector.processing').setLevel(logging.INFO)
logging.getLogger('src.core.m_detector.map_consistency').setLevel(logging.INFO) # MCC logs
logging.getLogger('src.core.m_detector.interpolation_utils').setLevel(logging.INFO) # Interpolation logs

# Add a stream handler to see logs in the notebook output if not already configured globally
# (You might already have this from your previous setup)
# stream_handler = logging.StreamHandler(sys.stdout)
# formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
# stream_handler.setFormatter(formatter)
# for logger_name in ['src.core.m_detector.map_consistency', 'src.core.m_detector.interpolation_utils']:
#     logger_to_add_handler = logging.getLogger(logger_name)
#     if not logger_to_add_handler.hasHandlers():
#         logger_to_add_handler.addHandler(stream_handler)
print("Logging configured (MCC and Interpolation set to DEBUG).")

# logging.getLogger('src.core.m_detector.map_consistency').setLevel(logging.DEBUG) # MCC logs
# logging.getLogger('src.core.m_detector.interpolation_utils').setLevel(logging.DEBUG) # Interpolation logs

# --- Load Configuration using Accessor ---
config_accessor: Optional[MDetectorConfigAccessor] = None
try:
    config_file_path = os.path.join(PROJECT_ROOT, 'config/m_detector_config.yaml')
    config_accessor = MDetectorConfigAccessor(config_file_path)
    print(f"Configuration loaded successfully from: {config_file_path}")
except Exception as e:
    print(f"Error loading config: {e}")
    raise # Stop if config fails

# --- Initialize NuScenes ---
# (Same as your Cell 1)
nusc: Optional[NuScenes] = None
if config_accessor:
    nuscenes_params = config_accessor.get_nuscenes_params()
    try:
        nusc = NuScenes(
            version=nuscenes_params.get('version', 'v1.0-mini'),
            dataroot=nuscenes_params.get('dataroot', '/datasets/nuscenes'),
            verbose=nuscenes_params.get('verbose_init', False)
        )
        print(f"NuScenes SDK initialized (Version: {nuscenes_params.get('version')}).")
    except Exception as e:
        print(f"Error initializing NuScenes SDK: {e}")
        raise
else:
    print("ERROR: MDetectorConfigAccessor failed. Cannot initialize NuScenes SDK.")
    raise ValueError("Config Accessor failed.")


# --- Scene and Sweep Selection for Debugging Interpolation ---
TARGET_SCENE_NAME_INTERP = 'scene-0103' # Choose a scene with interesting far points
# Number of sweeps to feed MDet *before* the one we'll pick a point from
SWEEPS_TO_PRIME_MDET_INTERP = 20
# Index of the target sweep *within the primed sequence* (0 is the first after priming)
# Or, an absolute index in the scene if you prefer, then adjust priming.
# Let's use an absolute index for clarity, and prime up to it.
TARGET_SWEEP_SCENE_INDEX_INTERP = 50 # e.g., the 51st sweep in the scene

target_scene_rec_interp: Optional[Dict[str, Any]] = None
TARGET_LIDAR_SD_TOKEN_INTERP: Optional[str] = None
target_sweep_data_dict_interp: Optional[Dict[str, Any]] = None
# This list will contain all sweeps up to and including the target sweep
sweeps_to_feed_for_interp_debug: List[Dict[str, Any]] = []

if nusc and config_accessor:
    scene_found = False
    for scene in nusc.scene:
        if scene['name'] == TARGET_SCENE_NAME_INTERP:
            target_scene_rec_interp = scene
            scene_found = True
            break
    if not scene_found:
        raise ValueError(f"ERROR: Scene '{TARGET_SCENE_NAME_INTERP}' not found.")
    
    print(f"Target Scene for Interpolation Debug: '{target_scene_rec_interp['name']}' (Token: {target_scene_rec_interp['token']})")
    all_sweeps_for_scene_interp = list(get_scene_sweep_data_sequence(nusc, target_scene_rec_interp['token']))
    
    if not (0 <= TARGET_SWEEP_SCENE_INDEX_INTERP < len(all_sweeps_for_scene_interp)):
        raise ValueError(f"Target sweep index {TARGET_SWEEP_SCENE_INDEX_INTERP} out of bounds for scene (len {len(all_sweeps_for_scene_interp)}).")

    # Feed sweeps up to and including the target sweep
    # Ensure we have enough history for the M-Detector library to be meaningful
    start_idx_feed = max(0, TARGET_SWEEP_SCENE_INDEX_INTERP - config_accessor.get_library_size() + 1) # Ensure library can be full
    
    sweeps_to_feed_for_interp_debug = all_sweeps_for_scene_interp[start_idx_feed : TARGET_SWEEP_SCENE_INDEX_INTERP + 1]
    
    if not sweeps_to_feed_for_interp_debug:
        raise ValueError("No sweeps selected to feed for interpolation debug.")
        
    target_sweep_data_dict_interp = sweeps_to_feed_for_interp_debug[-1]
    TARGET_LIDAR_SD_TOKEN_INTERP = target_sweep_data_dict_interp['lidar_sd_token']
    print(f"  Selected Target Sweep for analysis: Index {TARGET_SWEEP_SCENE_INDEX_INTERP} (SD Token: ...{TARGET_LIDAR_SD_TOKEN_INTERP[-8:]})")
    print(f"  Will feed {len(sweeps_to_feed_for_interp_debug)} sweeps to M-Detector (from scene index {start_idx_feed} to {TARGET_SWEEP_SCENE_INDEX_INTERP}).")

print("\nCell 1: Setup and Configuration for Interpolation Debug - OK")


Logging configured (MCC and Interpolation set to DEBUG).
Configuration loaded successfully from: /home/drugge/Unsupervised-Moving-Point-Detection/m_detector_python/config/m_detector_config.yaml
NuScenes SDK initialized (Version: v1.0-mini).
Target Scene for Interpolation Debug: 'scene-0103' (Token: fcbccedd61424f1b85dcbf8f897f9754)
  Selected Target Sweep for analysis: Index 50 (SD Token: ...83506d40)
  Will feed 20 sweeps to M-Detector (from scene index 31 to 50).

Cell 1: Setup and Configuration for Interpolation Debug - OK


In [4]:

# %% Cell 2: M-Detector Initialization
# (Same as your Cell 2)
detector_interp: Optional[MDetector] = None
if config_accessor:
    try:
        print("\nInitializing M-Detector for Interpolation Debug...")
        detector_interp = MDetector(config_accessor=config_accessor)
        print("M-Detector initialized successfully.")
    except Exception as e:
        print(f"Error initializing M-Detector: {e}")
        raise
else:
    raise ValueError("Config Accessor not available for M-Detector init.")
print("\nCell 2: M-Detector Initialization - OK")



INFO:src.core.m_detector.base.MDetector:Loaded adaptive_eps_config_occ_depth: {'enabled': True, 'dthr': 5.0, 'kthr': 0.01, 'dmax': 0.5, 'dmin': 0.05}
INFO:src.core.m_detector.base.MDetector:Loaded adaptive_eps_config_mc_fwd: {'enabled': True, 'dthr': 20.0, 'kthr': 0.07, 'dmax': 3.0, 'dmin': 0.2}
INFO:src.core.m_detector.base.MDetector:Loaded adaptive_eps_config_mc_bwd: {'enabled': True, 'dthr': 20.0, 'kthr': 0.03, 'dmax': 1.0, 'dmin': 0.05}
INFO:src.core.m_detector.base.MDetector:Event Test Config: Test1_N=10, Test1_M1=4, Test2_M2=2, Test3_M3=3
INFO:src.core.m_detector.base.MDetector:Detailed Occlusion Check Config: AngH_rad=0.005, AngV_rad=0.007, EpsDepth=0.10
INFO:src.core.m_detector.base.MDetector:MDetector initialized successfully.



Initializing M-Detector for Interpolation Debug...
M-Detector initialized successfully.

Cell 2: M-Detector Initialization - OK


In [5]:

# %% Cell 3: M-Detector Library Priming & Target DI Preparation
# This cell runs M-Detector on the selected sequence of sweeps.
# The last sweep added will be our `processed_target_di_interp`.
# Intermediate sweeps will have their labels processed by M-Detector.

processed_target_di_interp: Optional[DepthImage] = None
idx_of_processed_target_di_interp: Optional[int] = None

if detector_interp and target_scene_rec_interp and sweeps_to_feed_for_interp_debug and TARGET_LIDAR_SD_TOKEN_INTERP:
    print(f"\nProcessing sweeps for scene '{target_scene_rec_interp['name']}' to prime M-Detector library...")
    detector_interp.reset_scene_state()
    
    for i, sweep_data in enumerate(tqdm(sweeps_to_feed_for_interp_debug, desc="Feeding Sweeps to M-Detector (Interp Debug)")):
        added_di = detector_interp.add_sweep_and_create_depth_image(
            points_lidar_frame=sweep_data['points_sensor_frame'],
            T_global_lidar=sweep_data['T_global_lidar'],
            lidar_timestamp=sweep_data['timestamp'],
            lidar_sd_token=sweep_data['lidar_sd_token']
        )
        if not added_di:
            if sweep_data['lidar_sd_token'] == TARGET_LIDAR_SD_TOKEN_INTERP:
                raise ValueError(f"ERROR: Target sweep {TARGET_LIDAR_SD_TOKEN_INTERP} could not be added as DI.")
            continue

        # Process all DIs except potentially the very last one if we want to manually step through it
        if sweep_data['lidar_sd_token'] != TARGET_LIDAR_SD_TOKEN_INTERP:
            detector_interp.decide_and_process_frame(is_end_of_sequence=False)
        else: # This is our target sweep
            processed_target_di_interp = added_di
            # The target DI is added but NOT YET PROCESSED by decide_and_process_frame.
            # We will manually call process_and_label_di on it later, or just use it for MCC.
            # For MCC investigation, we need its index in the library.
            try:
                if detector_interp.depth_image_library and len(detector_interp.depth_image_library.get_all_images()) > 0:
                    idx_of_processed_target_di_interp = detector_interp.depth_image_library.get_all_images().index(processed_target_di_interp)
                    print(f"\nTarget sweep {TARGET_LIDAR_SD_TOKEN_INTERP[:8]}... added. Lib Idx: {idx_of_processed_target_di_interp}")
                else: idx_of_processed_target_di_interp = -1
            except ValueError: idx_of_processed_target_di_interp = -1
            
            # We can choose to process it here if we want its labels before MCC debug
            # For now, let's assume we might want to debug MCC on its raw state or after its own processing.
            # Let's process it to get its labels populated by the M-Detector logic.
            if idx_of_processed_target_di_interp != -1:
                 print(f"  Manually processing the target DI ({TARGET_LIDAR_SD_TOKEN_INTERP[:8]}) now...")
                 detector_interp.process_and_label_di(processed_target_di_interp, idx_of_processed_target_di_interp)
            else:
                 print(f"  ERROR: Target DI not found in library after adding.")


    if not processed_target_di_interp:
        raise ValueError(f"Target sweep {TARGET_LIDAR_SD_TOKEN_INTERP} was not processed.")
    if idx_of_processed_target_di_interp is None or idx_of_processed_target_di_interp == -1:
        raise ValueError("Target DI index in library could not be confirmed.")
else:
    raise ValueError("Prerequisites for M-Detector priming not met.")

print("\nCell 3: M-Detector Library Priming for Interpolation Debug - OK")


CRITICAL:src.core.m_detector.base.MDetector:!!!!!! MDetector.reset_scene_state CALLED. Library will be cleared. Current lib size: 0 !!!!!!



Processing sweeps for scene 'scene-0103' to prime M-Detector library...


Feeding Sweeps to M-Detector (Interp Debug):   0%|          | 0/20 [00:00<?, ?it/s]

/home/drugge/Unsupervised-Moving-Point-Detection/m_detector_python/src/core/m_detector/pre_labelers.py:85: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at /pytorch/aten/src/ATen/native/Cross.cpp:62.)
  normals = torch.cross(v1, v2)
INFO:src.core.m_detector.base.MDetector:RANSAC pre-labeled 15003 points as ground.
INFO:src.core.m_detector.base.MDetector:ADD_SWEEP: Added DI for TS 1533151605098086. Lib size now: 1. Max size: 20
INFO:src.core.m_detector.base.MDetector:DECIDE_FRAME_START: Lib len: 1, Min sweeps needed: 5, Last processed TS: None
INFO:src.core.m_detector.base.MDetector:RANSAC pre-labeled 15080 points as ground.
INFO:src.core.m_detector.base.MDetector:ADD_SWEEP: Added DI for TS 1533151605147845. Lib size now: 2. Max size: 20
INFO:src.core.m_detector.base.MDete


Target sweep 9181fe90... added. Lib Idx: 19
  Manually processing the target DI (9181fe90) now...


INFO:src.core.m_detector.base.MDetector:Processed DI (TS: 1533151606048630, Idx: 19): Labeled 22086/22086 points. Counts: {'OCCLUDING_IMAGE': 782, 'OCCLUDED_BY_IMAGE': 235, 'EMPTY_IN_IMAGE': 16, 'UNDETERMINED': 5825, 'PRELABELED_STATIC_GROUND': 15228}



Cell 3: M-Detector Library Priming for Interpolation Debug - OK


In [6]:
# %% Cell 3.5: Process Target DI With and Without Interpolation for Comparison (Corrected for Pre-labels and Trace Points)

import copy
import numpy as np # Ensure numpy is imported
import os
import pandas as pd # For easier analysis
import logging # Ensure logging is imported

from src.utils.debug_helpers import get_misclassified_points # Assuming debug_point_m_detector_logic is not directly used here
from src.core.constants import OcclusionResult # For OcclusionResult enum
from src.core.debug_collector import PointDebugCollector # For PointDebugCollector

# --- Logging Setup (adjust as needed) ---
logging.getLogger('src.core.m_detector.map_consistency').setLevel(logging.INFO)
logging.getLogger('src.core.m_detector.interpolation_utils').setLevel(logging.INFO)
# logging.getLogger('src.core.m_detector.base').setLevel(logging.DEBUG) # If you need MDetector base logs
# logging.getLogger('src.core.m_detector.processing').setLevel(logging.DEBUG) # If you need processing logs

# --- Ensure GT is loaded (Prerequisite from previous cell or earlier in this cell) ---
# This section assumes variables like nusc, target_scene_rec_interp, TARGET_LIDAR_SD_TOKEN_INTERP,
# processed_target_di_interp, idx_of_processed_target_di_interp, config_accessor, detector_interp,
# target_sweep_data_dict_interp, and PROJECT_ROOT are already defined from earlier parts of your notebook (e.g., Cell 3 setup).

gt_indices_dict_for_target_interp: Optional[Dict[str, Any]] = None # Initialize

if 'nusc' in locals() and 'target_scene_rec_interp' in locals() and \
   'TARGET_LIDAR_SD_TOKEN_INTERP' in locals() and 'processed_target_di_interp' in locals() and \
   processed_target_di_interp.original_points_global_coords is not None and \
   'config_accessor' in locals() and 'detector_interp' in locals() and \
   'idx_of_processed_target_di_interp' in locals() and 'target_sweep_data_dict_interp' in locals() and \
   'PROJECT_ROOT' in locals() :

    print(f"\nLoading Ground Truth labels for the target sweep (Token: {TARGET_LIDAR_SD_TOKEN_INTERP[:8]}) for analysis...")
    gt_labels_hdf5_dir_interp = config_accessor.get_nuscenes_params().get('label_path')
    if gt_labels_hdf5_dir_interp and not os.path.isabs(gt_labels_hdf5_dir_interp):
        gt_labels_hdf5_dir_interp = os.path.join(PROJECT_ROOT, gt_labels_hdf5_dir_interp)
    
    gt_labels_scene_hdf5_filepath_interp = os.path.join(gt_labels_hdf5_dir_interp, f"gt_point_labels_{target_scene_rec_interp['name']}.h5")
    
    validation_params_interp = config_accessor.get_validation_params()
    velocity_threshold_gt_interp = validation_params_interp.get('gt_velocity_threshold', 0.5)
    
    point_pre_filtering_params_interp = detector_interp.config_accessor.get_point_pre_filtering_params()
    mdet_min_range_interp = point_pre_filtering_params_interp.get('min_range_meters', 1.0)
    mdet_max_range_interp = point_pre_filtering_params_interp.get('max_range_meters', 80.0)

    if os.path.exists(gt_labels_scene_hdf5_filepath_interp):
        gt_indices_dict_for_target_interp = get_gt_dynamic_points_for_sweep(
            nusc=nusc,
            sweep_data_dict=target_sweep_data_dict_interp,
            all_points_global_mdetector=processed_target_di_interp.original_points_global_coords,
            gt_labels_scene_hdf5_path=gt_labels_scene_hdf5_filepath_interp,
            velocity_threshold=velocity_threshold_gt_interp,
            mdetector_min_range=mdet_min_range_interp,
            mdetector_max_range=mdet_max_range_interp
        )
        if not gt_indices_dict_for_target_interp or gt_indices_dict_for_target_interp.get('error_msg'):
            print(f"Error loading GT for target sweep: {gt_indices_dict_for_target_interp.get('error_msg', 'Unknown GT loading error')}")
            gt_indices_dict_for_target_interp = None # Ensure it's None on error
    else:
        print(f"ERROR: GT HDF5 file not found for interpolation debug: {gt_labels_scene_hdf5_filepath_interp}")
        gt_indices_dict_for_target_interp = None # Ensure it's None on error
else:
    print("One or more prerequisite variables for GT loading (nusc, target_scene_rec_interp, etc.) are not defined.")
    gt_indices_dict_for_target_interp = None # Ensure it's None

# --- Main Comparison Block ---
if 'detector_interp' in locals() and detector_interp is not None and \
   'processed_target_di_interp' in locals() and processed_target_di_interp is not None and \
   'idx_of_processed_target_di_interp' in locals() and idx_of_processed_target_di_interp is not None and \
   gt_indices_dict_for_target_interp is not None: # Check GT was loaded

    print("\n--- Comparing Target DI Processing: With vs. Without Interpolation (Detailed) ---")

    # --- 1. Save Original Pre-labels from the Target DI ---
    if processed_target_di_interp.mdet_labels_for_points is None:
        # This might happen if Cell 3 didn't run or target_di wasn't processed yet.
        # For this cell, we assume Cell 3 ran and processed_target_di_interp is the DI from the library.
        print("ERROR: processed_target_di_interp.mdet_labels_for_points is None. Re-run Cell 3 or ensure target DI is processed.")
        raise ValueError("Target DI has no initial labels.")
    
    original_target_di_labels_with_prelabels = processed_target_di_interp.mdet_labels_for_points.copy()
    num_pts_in_target_di = processed_target_di_interp.original_points_global_coords.shape[0]
    prelabeled_ground_mask = (original_target_di_labels_with_prelabels == OcclusionResult.PRELABELED_STATIC_GROUND.value)
    
    print(f"Captured original labels for target DI. Found {np.sum(prelabeled_ground_mask)} PRELABELED_STATIC_GROUND points.")
    if np.sum(prelabeled_ground_mask) == 0:
        print("WARNING: No PRELABELED_STATIC_GROUND points found in the original labels of the target DI. Check Cell 3 RANSAC output.")


    # --- 2. Generate Baseline Labels (WITHOUT Interpolation) to Identify FPs/FNs for Tracing ---
    print("\n--- Generating baseline labels (WITHOUT Interpolation) to identify FPs/FNs for tracing ---")
    detector_interp.mc_interp_enabled = False # Ensure interpolation is OFF for baseline

    # Prepare DI for baseline processing
    initial_labels_for_baseline = np.full(num_pts_in_target_di, OcclusionResult.UNDETERMINED.value, dtype=np.int8)
    initial_labels_for_baseline[prelabeled_ground_mask] = OcclusionResult.PRELABELED_STATIC_GROUND.value
    processed_target_di_interp.mdet_labels_for_points = initial_labels_for_baseline
    processed_target_di_interp.raw_occlusion_results_vs_history = None # Reset for fresh calculation
    if hasattr(processed_target_di_interp, 'mdet_scores_for_points'):
        processed_target_di_interp.mdet_scores_for_points = np.zeros(num_pts_in_target_di, dtype=np.float32)

    # Process for baseline (NO debug collector needed here, just getting labels)
    result_baseline_run = detector_interp.process_and_label_di(processed_target_di_interp, idx_of_processed_target_di_interp)
    labels_from_baseline_run = processed_target_di_interp.mdet_labels_for_points.copy() # Save the labels from this run
    
    # Get FPs/FNs from THIS baseline run
    misclassified_baseline = get_misclassified_points(
        processed_target_di_interp, # This DI now has labels_from_baseline_run
        gt_indices_dict_for_target_interp
    )
    fp_indices_baseline = misclassified_baseline.get('fp_dynamic', [])
    fn_indices_baseline = misclassified_baseline.get('fn_dynamic', [])
    points_to_trace_indices = sorted(list(set(fp_indices_baseline + fn_indices_baseline)))
    
    if not points_to_trace_indices:
        print("No FPs or FNs in baseline (without interp) run. Will trace a small random sample for debug store population.")
        if num_pts_in_target_di > 0:
            num_sample_trace = min(100, num_pts_in_target_di)
            points_to_trace_indices = list(np.random.choice(num_pts_in_target_di, num_sample_trace, replace=False))
        else:
            points_to_trace_indices = [] # Should not happen if DI has points
            
    print(f"Identified {len(fp_indices_baseline)} FPs and {len(fn_indices_baseline)} FNs from baseline run.")
    print(f"Will trace detailed logic for {len(points_to_trace_indices)} points in subsequent runs.")
    print(f"  Baseline run label counts: {result_baseline_run.get('label_counts')}")


    # --- 3. Setup Debug Collector and Store ---
    debug_data_store = {} # To store { "with_interp": {pt_idx: data}, "without_interp": {pt_idx: data} }

    # --- 4. Run 1: WITH Interpolation (Data Capture for Traced Points) ---
    print(f"\n--- Run 1: Processing WITH Interpolation (mc_interp_enabled = True) ---")
    detector_interp.mc_interp_enabled = True # Ensure interpolation is ON
    
    collector_run1 = PointDebugCollector(points_to_trace=points_to_trace_indices)
    detector_interp.set_debug_collector(collector_run1)

    # Prepare DI for Run 1
    initial_labels_for_run1 = np.full(num_pts_in_target_di, OcclusionResult.UNDETERMINED.value, dtype=np.int8)
    initial_labels_for_run1[prelabeled_ground_mask] = OcclusionResult.PRELABELED_STATIC_GROUND.value
    processed_target_di_interp.mdet_labels_for_points = initial_labels_for_run1
    processed_target_di_interp.raw_occlusion_results_vs_history = None
    if hasattr(processed_target_di_interp, 'mdet_scores_for_points'):
        processed_target_di_interp.mdet_scores_for_points = np.zeros(num_pts_in_target_di, dtype=np.float32)

    result_with_interp_run = detector_interp.process_and_label_di(processed_target_di_interp, idx_of_processed_target_di_interp)
    labels_with_interp_final = processed_target_di_interp.mdet_labels_for_points.copy() # Save labels from this run
    debug_data_store["with_interp"] = copy.deepcopy(collector_run1.debug_data)
    detector_interp.set_debug_collector(None) # Clear collector
    collector_run1.clear_data() # Clear data from collector object
    print(f"Processing WITH interpolation complete. Collected trace for {len(debug_data_store['with_interp'])} points.")
    print(f"  Label counts: {result_with_interp_run.get('label_counts')}")


    # --- 5. Run 2: WITHOUT Interpolation (Data Capture for Traced Points - for direct comparison) ---
    print(f"\n--- Run 2: Processing WITHOUT Interpolation (mc_interp_enabled = False) ---")
    original_interp_setting_for_restore = detector_interp.mc_interp_enabled # Should be True from previous run
    detector_interp.mc_interp_enabled = False # Ensure interpolation is OFF
    
    collector_run2 = PointDebugCollector(points_to_trace=points_to_trace_indices) # Use same points
    detector_interp.set_debug_collector(collector_run2)

    # Prepare DI for Run 2
    initial_labels_for_run2 = np.full(num_pts_in_target_di, OcclusionResult.UNDETERMINED.value, dtype=np.int8)
    initial_labels_for_run2[prelabeled_ground_mask] = OcclusionResult.PRELABELED_STATIC_GROUND.value
    processed_target_di_interp.mdet_labels_for_points = initial_labels_for_run2
    processed_target_di_interp.raw_occlusion_results_vs_history = None
    if hasattr(processed_target_di_interp, 'mdet_scores_for_points'):
        processed_target_di_interp.mdet_scores_for_points = np.zeros(num_pts_in_target_di, dtype=np.float32)

    result_without_interp_run = detector_interp.process_and_label_di(processed_target_di_interp, idx_of_processed_target_di_interp)
    labels_without_interp_final = processed_target_di_interp.mdet_labels_for_points.copy() # Save labels
    debug_data_store["without_interp"] = copy.deepcopy(collector_run2.debug_data)
    detector_interp.set_debug_collector(None) # Clear collector
    collector_run2.clear_data() # Clear data from collector object
    
    detector_interp.mc_interp_enabled = original_interp_setting_for_restore # Restore to what it was before this run
    print(f"Processing WITHOUT interpolation complete. Collected trace for {len(debug_data_store['without_interp'])} points.")
    print(f"  Label counts: {result_without_interp_run.get('label_counts')}")
    print(f"Restored interpolation setting to: {detector_interp.mc_interp_enabled}")


    # --- 6. Metrics Comparison (using the labels from these controlled runs: Run 1 vs Run 2) ---
    print("\n--- Metrics Comparison (from controlled runs) ---")
    # Metrics for "With Interpolation" (using labels_with_interp_final from Run 1)
    processed_target_di_interp.mdet_labels_for_points = labels_with_interp_final
    misclassified_with_interp_ctrl = get_misclassified_points(processed_target_di_interp, gt_indices_dict_for_target_interp)
    fp_with_interp_ctrl = len(misclassified_with_interp_ctrl.get('fp_dynamic', []))
    fn_with_interp_ctrl = len(misclassified_with_interp_ctrl.get('fn_dynamic', []))
    print(f"\nResults WITH Interpolation (Controlled Run 1):")
    print(f"  False Positives (Static -> Dynamic): {fp_with_interp_ctrl}")
    print(f"  False Negatives (Dynamic -> Static/Undet): {fn_with_interp_ctrl}")

    # Metrics for "Without Interpolation" (using labels_without_interp_final from Run 2)
    processed_target_di_interp.mdet_labels_for_points = labels_without_interp_final
    misclassified_without_interp_ctrl = get_misclassified_points(processed_target_di_interp, gt_indices_dict_for_target_interp)
    fp_without_interp_ctrl = len(misclassified_without_interp_ctrl.get('fp_dynamic', []))
    fn_without_interp_ctrl = len(misclassified_without_interp_ctrl.get('fn_dynamic', []))
    print(f"\nResults WITHOUT Interpolation (Controlled Run 2):")
    print(f"  False Positives (Static -> Dynamic): {fp_without_interp_ctrl}")
    print(f"  False Negatives (Dynamic -> Static/Undet): {fn_without_interp_ctrl}")
    
    print("\n--- Differences (Controlled Runs: Run 1 With vs Run 2 Without) ---")
    print(f"  FP Change (With - Without): {fp_with_interp_ctrl - fp_without_interp_ctrl}")
    print(f"  FN Change (With - Without): {fn_with_interp_ctrl - fn_without_interp_ctrl}")

    # --- 7. Detailed Analysis of Traced Points Where Final Label Differs (between Run 1 and Run 2) ---
    print("\n--- Detailed Analysis of Traced Points Where Final Label Differs (Run 1 vs Run 2) ---")
    changed_due_to_interp_count = 0
    if not points_to_trace_indices:
        print("No points were specifically traced (points_to_trace_indices is empty).")
    else:
        for pt_idx in points_to_trace_indices:
            data_with = debug_data_store.get("with_interp", {}).get(pt_idx)
            data_without = debug_data_store.get("without_interp", {}).get(pt_idx)

            if data_with and data_without:
                final_label_w = data_with.get("final_label")
                final_label_wo = data_without.get("final_label")
                
                if final_label_w != final_label_wo:
                    changed_due_to_interp_count += 1
                    
                    gt_status_str = "Unknown_GT"
                    if pt_idx in gt_indices_dict_for_target_interp.get('gt_dynamic_indices', []): gt_status_str = "GT_Dynamic"
                    elif pt_idx in gt_indices_dict_for_target_interp.get('gt_static_indices', []): gt_status_str = "GT_Static"
                    elif pt_idx in gt_indices_dict_for_target_interp.get('unlabeled_indices', []): gt_status_str = "GT_Unlabeled"
                        
                    print(f"\nPoint IDX: {pt_idx} ({gt_status_str})")
                    print(f"  WITH INTERP (Run 1): Label: {final_label_w}")
                    # Add more details from data_with if needed for these differing points
                    print(f"    T1_MCC: {data_with.get('T1_mcc_result_bool')} (Reason: {data_with.get('T1_mcc_full_debug',{}).get('reason_for_result','N/A')[:100]}...)")
                    print(f"    T3_MCC: {data_with.get('T3_mcc_on_P_result_bool')} (Reason: {data_with.get('T3_mcc_on_P_full_debug',{}).get('reason_for_result','N/A')[:100]}...)")
                    if "MCC_Refinement_performed" in data_with:
                        print(f"    MCC_Refine: Perf={data_with.get('MCC_Refinement_performed')}, Res={data_with.get('MCC_Refinement_result_bool')}, Override={data_with.get('MCC_Refinement_overrode_to_static')}")

                    print(f"  WITHOUT INTERP (Run 2): Label: {final_label_wo}")
                    print(f"    T1_MCC: {data_without.get('T1_mcc_result_bool')} (Reason: {data_without.get('T1_mcc_full_debug',{}).get('reason_for_result','N/A')[:100]}...)")
                    print(f"    T3_MCC: {data_without.get('T3_mcc_on_P_result_bool')} (Reason: {data_without.get('T3_mcc_on_P_full_debug',{}).get('reason_for_result','N/A')[:100]}...)")
                    if "MCC_Refinement_performed" in data_without:
                         print(f"    MCC_Refine: Perf={data_without.get('MCC_Refinement_performed')}, Res={data_without.get('MCC_Refinement_result_bool')}, Override={data_without.get('MCC_Refinement_overrode_to_static')}")
            elif not data_with:
                print(f"Warning: No debug data for point {pt_idx} in 'with_interp' run.")
            elif not data_without:
                print(f"Warning: No debug data for point {pt_idx} in 'without_interp' run.")


    print(f"\nFound {changed_due_to_interp_count} traced points where the final label differed between Run 1 (With Interp) and Run 2 (Without Interp).")
    
    # --- 8. Restore original DI state ---
    # This ensures that processed_target_di_interp is back to its state after Cell 3 for any subsequent cells.
    processed_target_di_interp.mdet_labels_for_points = original_target_di_labels_with_prelabels
    # If raw_occlusion_results_vs_history was populated by Cell 3 and is needed by later cells,
    # you might need to save and restore it too. For now, assuming it's okay if it's None
    # and process_and_label_di will recalculate if needed.
    print("\nRestored original pre-labeled state (from Cell 3) to processed_target_di_interp.")

else:
    print("Comparison prerequisites not met (detector, target DI, GT labels, or other necessary variables missing).")
    # Define dummy variables so subsequent cells don't immediately error if they expect these
    debug_data_store = {}
    points_to_trace_indices = []
    misclassified_baseline = {} # Ensure this is defined for Cell 3.6
    fp_indices_baseline = []    # Ensure this is defined for Cell 3.6


# --- Make sure key variables are available for subsequent cells (like 3.6, 3.7, 3.8) ---
# misclassified_baseline should now be correctly defined from the initial baseline run within this cell.
# points_to_trace_indices is also defined.
# debug_data_store contains the detailed traces for "with_interp" and "without_interp" (Run 1 and Run 2).
# gt_indices_dict_for_target_interp holds the GT info.


Loading Ground Truth labels for the target sweep (Token: 9181fe90) for analysis...

--- Comparing Target DI Processing: With vs. Without Interpolation (Detailed) ---
Captured original labels for target DI. Found 15228 PRELABELED_STATIC_GROUND points.

--- Generating baseline labels (WITHOUT Interpolation) to identify FPs/FNs for tracing ---


INFO:src.core.m_detector.base.MDetector:Processed DI (TS: 1533151606048630, Idx: 19): Labeled 22086/22086 points. Counts: {'OCCLUDING_IMAGE': 799, 'OCCLUDED_BY_IMAGE': 233, 'EMPTY_IN_IMAGE': 16, 'UNDETERMINED': 5810, 'PRELABELED_STATIC_GROUND': 15228}


Identified 261 FPs and 248 FNs from baseline run.
Will trace detailed logic for 509 points in subsequent runs.
  Baseline run label counts: {0: 799, 1: 233, 2: 16, 3: 5810, 4: 15228}

--- Run 1: Processing WITH Interpolation (mc_interp_enabled = True) ---


INFO:src.core.m_detector.base.MDetector:Processed DI (TS: 1533151606048630, Idx: 19): Labeled 22086/22086 points. Counts: {'OCCLUDING_IMAGE': 782, 'OCCLUDED_BY_IMAGE': 235, 'EMPTY_IN_IMAGE': 16, 'UNDETERMINED': 5825, 'PRELABELED_STATIC_GROUND': 15228}


Processing WITH interpolation complete. Collected trace for 509 points.
  Label counts: {0: 782, 1: 235, 2: 16, 3: 5825, 4: 15228}

--- Run 2: Processing WITHOUT Interpolation (mc_interp_enabled = False) ---


INFO:src.core.m_detector.base.MDetector:Processed DI (TS: 1533151606048630, Idx: 19): Labeled 22086/22086 points. Counts: {'OCCLUDING_IMAGE': 799, 'OCCLUDED_BY_IMAGE': 233, 'EMPTY_IN_IMAGE': 16, 'UNDETERMINED': 5810, 'PRELABELED_STATIC_GROUND': 15228}


Processing WITHOUT interpolation complete. Collected trace for 509 points.
  Label counts: {0: 799, 1: 233, 2: 16, 3: 5810, 4: 15228}
Restored interpolation setting to: True

--- Metrics Comparison (from controlled runs) ---

Results WITH Interpolation (Controlled Run 1):
  False Positives (Static -> Dynamic): 244
  False Negatives (Dynamic -> Static/Undet): 248

Results WITHOUT Interpolation (Controlled Run 2):
  False Positives (Static -> Dynamic): 261
  False Negatives (Dynamic -> Static/Undet): 248

--- Differences (Controlled Runs: Run 1 With vs Run 2 Without) ---
  FP Change (With - Without): -17
  FN Change (With - Without): 0

--- Detailed Analysis of Traced Points Where Final Label Differs (Run 1 vs Run 2) ---

Point IDX: 19749 (GT_Unlabeled)
  WITH INTERP (Run 1): Label: UNDETERMINED
    T1_MCC: None (Reason: N/A...)
    T3_MCC: True (Reason: Consistent static points found in 5/9 DIs where checks were performed. Mode: Count, Threshold: 2. Me...)
    MCC_Refine: Perf=False, Re

In [7]:
# Cell 3.6

changed_mcc_results_count = 0
points_with_changed_mcc_outcome = []

for pt_idx in points_to_trace_indices: # Assumes this list is defined
    data_w = debug_data_store.get("with_interp", {}).get(pt_idx)
    data_wo = debug_data_store.get("without_interp", {}).get(pt_idx)

    if not data_w or not data_wo:
        continue

    # Check Test 1 MCC
    t1_mcc_w = data_w.get('T1_mcc_result_bool')
    t1_mcc_wo = data_wo.get('T1_mcc_result_bool')
    t1_mcc_performed_w = data_w.get('T1_mcc_performed')
    t1_mcc_performed_wo = data_wo.get('T1_mcc_performed')

    # Check Test 3 MCC (on P)
    t3_mcc_w = data_w.get('T3_mcc_on_P_result_bool')
    t3_mcc_wo = data_wo.get('T3_mcc_on_P_result_bool')
    
    # Check if MCC was performed in both and results differ, or performed in one and not other
    # (More nuanced check might be needed if mcc_performed itself differs)
    
    mcc_outcome_changed = False
    if t1_mcc_performed_w and t1_mcc_performed_wo and t1_mcc_w != t1_mcc_wo:
        mcc_outcome_changed = True
        print(f"\nPt {pt_idx}: T1_MCC outcome changed. With: {t1_mcc_w}, Without: {t1_mcc_wo}")
        if data_w.get('T1_mcc_full_debug'):
            print(f"  Reason With: {data_w['T1_mcc_full_debug'].get('reason_for_result', 'N/A')}")
            print(f"    Interp Depth With: {data_w['T1_mcc_full_debug'].get('relevant_dis_details', [{}])[0].get('interpolation_result_depth', 'N/A') if data_w['T1_mcc_full_debug'].get('relevant_dis_details') else 'N/A'}")
        if data_wo.get('T1_mcc_full_debug'):
            print(f"  Reason Without: {data_wo['T1_mcc_full_debug'].get('reason_for_result', 'N/A')}")


    if t3_mcc_w is not None and t3_mcc_wo is not None and t3_mcc_w != t3_mcc_wo : # MCC for T3 is always attempted if enabled
        mcc_outcome_changed = True
        print(f"\nPt {pt_idx}: T3_MCC_on_P outcome changed. With: {t3_mcc_w}, Without: {t3_mcc_wo}")
        if data_w.get('T3_mcc_on_P_full_debug'):
            print(f"  Reason With: {data_w['T3_mcc_on_P_full_debug'].get('reason_for_result', 'N/A')}")
            print(f"    Interp Depth With: {data_w['T3_mcc_on_P_full_debug'].get('relevant_dis_details', [{}])[0].get('interpolation_result_depth', 'N/A') if data_w['T3_mcc_on_P_full_debug'].get('relevant_dis_details') else 'N/A'}")
        if data_wo.get('T3_mcc_on_P_full_debug'):
             print(f"  Reason Without: {data_wo['T3_mcc_on_P_full_debug'].get('reason_for_result', 'N/A')}")

    if mcc_outcome_changed:
        changed_mcc_results_count += 1
        points_with_changed_mcc_outcome.append(pt_idx)
        # Now check if this differing MCC outcome led to different final labels
        if data_w.get("final_label") != data_wo.get("final_label"):
            print(f"  >>> And FINAL LABEL CHANGED: With: {data_w.get('final_label')}, Without: {data_wo.get('final_label')}")
        else:
            print(f"  >>> But FINAL LABEL DID NOT CHANGE. With: {data_w.get('final_label')}, Without: {data_wo.get('final_label')}")
            print(f"      Intermediate outcomes WITH: T1/4={data_w.get('OUTCOME_T1_T4')}, T2={data_w.get('OUTCOME_T2')}, T3={data_w.get('OUTCOME_T3')}")
            print(f"      Intermediate outcomes WITHOUT: T1/4={data_wo.get('OUTCOME_T1_T4')}, T2={data_wo.get('OUTCOME_T2')}, T3={data_wo.get('OUTCOME_T3')}")


print(f"\nFound {changed_mcc_results_count} traced points where an MCC boolean outcome differed.")
if not points_with_changed_mcc_outcome:
    print("No traced points had differing MCC outcomes. This implies interpolation isn't changing the core MCC decision for these FPs/FNs.")

# %% Cell 3.6: Detailed Analysis of FP Points and MCC Test 1 Behavior

import pandas as pd # Optional, but can be nice for viewing lots of data

if 'debug_data_store' not in locals() or \
   'points_to_trace_indices' not in locals() or \
   'gt_indices_dict_for_target_interp' not in locals() or \
   'misclassified_baseline' not in locals() : # misclassified_baseline from Cell 3.5
    print("ERROR: Prerequisite data from Cell 3.5 (debug_data_store, points_to_trace_indices, gt_indices_dict_for_target_interp, misclassified_baseline) not found.")
    print("Please ensure Cell 3.5 ran successfully and defined these variables.")
else:
    print("\n--- A.1: Analyzing False Positives (FPs) and Test 1 MCC Behavior ---")

    fp_indices_baseline = set(misclassified_baseline.get('fp_dynamic', []))
    # We are interested in points that were FPs in the baseline (without interpolation) run.
    
    # For determining GT status easily
    gt_dynamic_indices_set = set(gt_indices_dict_for_target_interp.get('gt_dynamic_indices', []))
    gt_static_indices_set = set(gt_indices_dict_for_target_interp.get('gt_static_indices', []))
    gt_unlabeled_indices_set = set(gt_indices_dict_for_target_interp.get('unlabeled_indices', []))

    analysis_results = []
    changed_fp_to_static_count = 0
    fp_mcc_changed_but_label_not_static = 0
    fp_mcc_no_change_remained_fp = 0

    for pt_idx in points_to_trace_indices:
        if pt_idx not in fp_indices_baseline:
            continue # Only analyze points that were FPs in the baseline

        data_w = debug_data_store.get("with_interp", {}).get(pt_idx)
        data_wo = debug_data_store.get("without_interp", {}).get(pt_idx)

        if not data_w or not data_wo:
            print(f"Warning: Missing trace data for FP pt_idx {pt_idx}")
            continue

        # --- Basic Info ---
        gt_status_str = "Unknown"
        if pt_idx in gt_dynamic_indices_set: gt_status_str = "GT_Dynamic"
        elif pt_idx in gt_static_indices_set: gt_status_str = "GT_Static"
        elif pt_idx in gt_unlabeled_indices_set: gt_status_str = "GT_Unlabeled"
        
        # Sanity check: this point should be an FP in the 'without' run
        if data_wo.get("final_label") != "OCCLUDING_IMAGE": # Assuming OCCLUDING_IMAGE is your dynamic label
            # This might happen if the definition of FP in get_misclassified_points differs
            # or if points_to_trace_indices wasn't purely from baseline FPs.
            # print(f"Debug: Pt {pt_idx} was in fp_indices_baseline, but its 'without_interp' label is {data_wo.get('final_label')}. GT: {gt_status_str}. Skipping detailed FP analysis for this point.")
            pass # Continue if it's not an FP as per the detailed trace for "without" run

        point_analysis = {
            "pt_idx": pt_idx,
            "gt_status": gt_status_str,
            "raw_occ_imm_hist_wo": data_wo.get("raw_occ_imm_hist"),
            "t1_mcc_performed_wo": data_wo.get("T1_mcc_performed"),
            "t1_mcc_result_bool_wo": data_wo.get("T1_mcc_result_bool"),
            "t1_outcome_after_mcc_wo": data_wo.get("T1_outcome_after_mcc"),
            "final_label_wo": data_wo.get("final_label"),
            
            "raw_occ_imm_hist_w": data_w.get("raw_occ_imm_hist"),
            "t1_mcc_performed_w": data_w.get("T1_mcc_performed"),
            "t1_mcc_result_bool_w": data_w.get("T1_mcc_result_bool"),
            "t1_outcome_after_mcc_w": data_w.get("T1_outcome_after_mcc"),
            "final_label_w": data_w.get("final_label"),
            "interp_changed_t1_mcc_bool": None,
            "interp_corrected_fp": None,
            "mcc_reason_with_interp_t1": "N/A",
            "mcc_reason_without_interp_t1": "N/A"
        }

        # --- Focus on Test 1 MCC path for these FPs ---
        # An FP means GT is static/unlabeled, but MDet (without interp) labeled it dynamic (OCCLUDING_IMAGE).
        # For this to happen via Test 1/4, raw_occ_imm_hist was likely OCCLUDING_IMAGE,
        # and T1_mcc_result_bool_wo was False (map inconsistent).
        
        if data_wo.get("raw_occ_imm_hist") == "OCCLUDING_IMAGE":
            point_analysis["t1_path_active_for_fp_wo"] = True
            
            if data_wo.get("T1_mcc_full_debug"):
                point_analysis["mcc_reason_without_interp_t1"] = data_wo["T1_mcc_full_debug"].get('reason_for_result', 'N/A')

            if data_w.get("T1_mcc_full_debug"):
                 point_analysis["mcc_reason_with_interp_t1"] = data_w["T1_mcc_full_debug"].get('reason_for_result', 'N/A')


            if data_w.get("T1_mcc_result_bool") != data_wo.get("T1_mcc_result_bool"):
                point_analysis["interp_changed_t1_mcc_bool"] = True
            else:
                point_analysis["interp_changed_t1_mcc_bool"] = False

            # Did interpolation help correct this FP?
            # Corrected means: final_label_w is OCCLUDED_BY_IMAGE or UNDETERMINED (i.e., not OCCLUDING_IMAGE)
            # AND final_label_wo was OCCLUDING_IMAGE
            if data_wo.get("final_label") == "OCCLUDING_IMAGE" and \
               data_w.get("final_label") != "OCCLUDING_IMAGE":
                point_analysis["interp_corrected_fp"] = True
                changed_fp_to_static_count +=1
            else:
                point_analysis["interp_corrected_fp"] = False
                if point_analysis["interp_changed_t1_mcc_bool"] and data_w.get("final_label") == "OCCLUDING_IMAGE":
                    fp_mcc_changed_but_label_not_static += 1
                elif not point_analysis["interp_changed_t1_mcc_bool"] and data_w.get("final_label") == "OCCLUDING_IMAGE":
                    fp_mcc_no_change_remained_fp +=1
        else:
            # This FP was labeled dynamic through another path (e.g. Test 2 or Test 4 override)
            point_analysis["t1_path_active_for_fp_wo"] = False
            # It's less direct to see interp's effect here via T1_MCC, but T3_MCC might be relevant.
            # For now, focusing on T1_MCC for FPs.

        analysis_results.append(point_analysis)

    # --- Summarize Findings for FPs ---
    print(f"\n--- Summary of FP Analysis (Total FPs in baseline: {len(fp_indices_baseline)}) ---")
    
    if not analysis_results:
        print("No FPs were processed in the detailed analysis loop (e.g. points_to_trace_indices was empty or only contained FNs).")
    else:
        df_fp_analysis = pd.DataFrame(analysis_results)
        
        active_t1_path_fps = df_fp_analysis[df_fp_analysis["t1_path_active_for_fp_wo"] == True]
        print(f"Number of baseline FPs where T1 path (raw_occ=OCCLUDING_IMAGE) was active: {len(active_t1_path_fps)}")

        if not active_t1_path_fps.empty:
            interp_changed_mcc_for_fp = active_t1_path_fps[active_t1_path_fps["interp_changed_t1_mcc_bool"] == True]
            print(f"  Out of these, number of FPs where interpolation CHANGED T1_MCC boolean outcome: {len(interp_changed_mcc_for_fp)}")
            
            corrected_fps_via_t1_mcc = interp_changed_mcc_for_fp[interp_changed_mcc_for_fp["interp_corrected_fp"] == True]
            print(f"    Number of FPs CORRECTED to non-dynamic by interp via T1_MCC change: {len(corrected_fps_via_t1_mcc)}")

            if len(corrected_fps_via_t1_mcc) > 0:
                print("\n    Examples of FPs corrected by Interpolation via T1_MCC:")
                for _, row in corrected_fps_via_t1_mcc.head(5).iterrows(): # Print first 5 examples
                    print(f"      Pt {row['pt_idx']}: GT: {row['gt_status']}")
                    print(f"        T1_MCC_WO: {row['t1_mcc_result_bool_wo']} (Reason: {row['mcc_reason_without_interp_t1'][:100]}...) -> Final_WO: {row['final_label_wo']}")
                    print(f"        T1_MCC_W:  {row['t1_mcc_result_bool_w']} (Reason: {row['mcc_reason_with_interp_t1'][:100]}...) -> Final_W: {row['final_label_w']}")
            
            interp_no_change_mcc_for_fp = active_t1_path_fps[active_t1_path_fps["interp_changed_t1_mcc_bool"] == False]
            print(f"\n  Out of active T1 path FPs, number of FPs where interpolation DID NOT change T1_MCC boolean outcome: {len(interp_no_change_mcc_for_fp)}")
            if len(interp_no_change_mcc_for_fp) > 0:
                print("    (These FPs remained FPs, and T1_MCC gave same result for both interp on/off)")
                # You could print examples here too, focusing on their T1_mcc_full_debug reasons.
                # for _, row in interp_no_change_mcc_for_fp.head(3).iterrows():
                #     print(f"      Pt {row['pt_idx']}: T1_MCC_W reason: {row['mcc_reason_with_interp_t1']}")


        print(f"\nTotal FPs where interp changed T1_MCC outcome but final label remained OCCLUDING_IMAGE (possibly due to T2/T3/T4 override): {fp_mcc_changed_but_label_not_static}")
        print(f"Total FPs where T1_MCC outcome did not change AND final label remained OCCLUDING_IMAGE: {fp_mcc_no_change_remained_fp}")
        print(f"Overall count of FPs that became non-dynamic with interpolation: {changed_fp_to_static_count}")
        
# %% Cell 3.7: Identify Source of False Positive Labels (Without Interpolation Run)

if 'debug_data_store' not in locals() or \
   'points_to_trace_indices' not in locals() or \
   'gt_indices_dict_for_target_interp' not in locals() or \
   'misclassified_baseline' not in locals() :
    print("ERROR: Prerequisite data from Cell 3.5 not found.")
else:
    print("\n--- Analyzing Source of FP Labels (from 'Without Interpolation' Run) ---")

    fp_indices_baseline = set(misclassified_baseline.get('fp_dynamic', []))
    
    fp_source_counts = {
        "by_test1_raw_immediate": 0, # Should be 0 based on previous output
        "by_test4_perp_event": 0,
        "by_test2_parallel": 0,
        "by_test3_appearance": 0,
        "by_multiple_tests": 0, # If more than one test flagged it
        "unknown_source_but_fp": 0
    }

    fp_details_for_investigation = [] # List of dicts for points of interest

    for pt_idx in points_to_trace_indices:
        if pt_idx not in fp_indices_baseline:
            continue

        data_wo = debug_data_store.get("without_interp", {}).get(pt_idx)
        if not data_wo:
            continue

        # We are only interested if the final label in the 'without' run was OCCLUDING_IMAGE
        if data_wo.get("final_label") != "OCCLUDING_IMAGE":
            continue

        is_dynamic_by_t1_raw_imm = data_wo.get("raw_occ_imm_hist") == "OCCLUDING_IMAGE" and \
                                   data_wo.get("T1_mcc_result_bool") == False # MCC did not make it static
        
        is_dynamic_by_t4_perp = data_wo.get("T4_perp_event_passed") == True
        
        # final_outcome_tests1_and_4 is OCCLUDING_IMAGE if either T4_perp_event_passed is True,
        # OR (T4_perp_event_passed is False AND T1_outcome_after_mcc is OCCLUDING_IMAGE)
        outcome_t1_t4_is_dynamic = data_wo.get("OUTCOME_T1_T4") == "OCCLUDING_IMAGE"
        
        outcome_t2_is_dynamic = data_wo.get("OUTCOME_T2") == "OCCLUDING_IMAGE" # From Test2_passed
        outcome_t3_is_dynamic = data_wo.get("OUTCOME_T3") == "OCCLUDING_IMAGE" # From Test3_chain_passed (and T3_mcc_on_P was False)

        dynamic_sources = []
        if outcome_t1_t4_is_dynamic: dynamic_sources.append("T1/T4")
        if outcome_t2_is_dynamic: dynamic_sources.append("T2")
        if outcome_t3_is_dynamic: dynamic_sources.append("T3")

        # Refined counting based on the final aggregation logic
        if outcome_t1_t4_is_dynamic: # This is the result of (raw_occ_imm + MCC for T1) OR (T4 perp event)
            # If T4 was the cause
            if is_dynamic_by_t4_perp:
                fp_source_counts["by_test4_perp_event"] +=1
            # If T1 raw + MCC was the cause (and T4 was not)
            elif data_wo.get("T1_outcome_after_mcc") == "OCCLUDING_IMAGE": # T1_raw was OCCLUDING and MCC didn't change it
                 fp_source_counts["by_test1_raw_immediate"] +=1 # This should still be 0 based on your output
        
        if outcome_t2_is_dynamic:
            fp_source_counts["by_test2_parallel"] += 1
        
        if outcome_t3_is_dynamic:
            fp_source_counts["by_test3_appearance"] += 1

        # Note: The above counts might double-count if a point is flagged by multiple tests.
        # The final label is an OR. Let's count how many FPs were triggered by each specific test path
        # assuming other paths might also have triggered it.
        
        if len(dynamic_sources) > 1:
            fp_source_counts["by_multiple_tests"] += 1 # This is just an indicator
            
        if not dynamic_sources: # Should not happen if final_label is OCCLUDING_IMAGE
            fp_source_counts["unknown_source_but_fp"] +=1
            
        # Store details for a few FPs from each dominant category later if needed
        fp_details_for_investigation.append({
            "pt_idx": pt_idx,
            "dynamic_sources": dynamic_sources,
            "T1_raw_occ_imm_hist_wo": data_wo.get("raw_occ_imm_hist"),
            "T1_mcc_performed_wo": data_wo.get("T1_mcc_performed"),
            "T1_mcc_result_bool_wo": data_wo.get("T1_mcc_result_bool"),
            "T4_perp_event_passed_wo": data_wo.get("T4_perp_event_passed"),
            "OUTCOME_T1_T4_wo": data_wo.get("OUTCOME_T1_T4"),
            "Test2_passed_wo": data_wo.get("Test2_passed"),
            "OUTCOME_T2_wo": data_wo.get("OUTCOME_T2"),
            "T3_mcc_on_P_result_bool_wo": data_wo.get("T3_mcc_on_P_result_bool"),
            "Test3_chain_passed_wo": data_wo.get("Test3_chain_passed"),
            "OUTCOME_T3_wo": data_wo.get("OUTCOME_T3"),
        })


    print("\n--- Counts of FPs Triggered by Each Test Path ('Without Interpolation' run) ---")
    # These counts might sum to more than total FPs if points are caught by multiple tests.
    # The final label is dynamic if ANY of these paths make it dynamic.
    print(f"  FPs where OUTCOME_T1_T4 was OCCLUDING_IMAGE: {sum(1 for d in fp_details_for_investigation if d['OUTCOME_T1_T4_wo'] == 'OCCLUDING_IMAGE')}")
    print(f"    Breakdown: FPs by T4 (perp event passed): {sum(1 for d in fp_details_for_investigation if d['T4_perp_event_passed_wo'] == True)}")
    print(f"    Breakdown: FPs by T1_raw+MCC (and T4 failed): {sum(1 for d in fp_details_for_investigation if d['T4_perp_event_passed_wo'] == False and d['OUTCOME_T1_T4_wo'] == 'OCCLUDING_IMAGE')}")
    print(f"  FPs where OUTCOME_T2 (parallel) was OCCLUDING_IMAGE: {sum(1 for d in fp_details_for_investigation if d['OUTCOME_T2_wo'] == 'OCCLUDING_IMAGE')}")
    print(f"  FPs where OUTCOME_T3 (appearance) was OCCLUDING_IMAGE: {sum(1 for d in fp_details_for_investigation if d['OUTCOME_T3_wo'] == 'OCCLUDING_IMAGE')}")
    
    if fp_source_counts["unknown_source_but_fp"] > 0:
        print(f"  WARNING: Found {fp_source_counts['unknown_source_but_fp']} FPs whose dynamic source couldn't be identified by this script (logic error?).")

# %% Cell 3.8: Deep Dive into FPs (MCC Refinement Failure) & FN Categorization

import pandas as pd

if 'debug_data_store' not in locals() or \
   'misclassified_baseline' not in locals() or \
   'points_to_trace_indices' not in locals() or \
   'gt_indices_dict_for_target_interp' not in locals():
    print("ERROR: Prerequisite data not found. Ensure cells up to 3.7 (or equivalent) have run successfully with the MCC Refinement logic.")
else:
    print("\n--- Deep Dive into FPs (Why MCC Refinement Failed) & FN Categorization ---")

    # --- Setup: Get FP and FN indices from the 'with_interp' run ---
    # We need to re-calculate misclassified points based on the 'with_interp' run's output
    # if the MCC refinement step was part of it.
    # For simplicity, let's assume misclassified_baseline was generated from a run
    # that already included the MCC refinement. If not, you'd need to regenerate it.
    # The key is that debug_data_store["with_interp"] has the MCC_Refinement_xxx fields.

    fp_indices_current_run = set()
    fn_indices_current_run = set()
    
    # Assuming 'gt_indices_dict_for_target_interp' has 'gt_static_indices' and 'gt_dynamic_indices'
    gt_static_indices_set = set(gt_indices_dict_for_target_interp.get('gt_static_indices', []))
    gt_dynamic_indices_set = set(gt_indices_dict_for_target_interp.get('gt_dynamic_indices', []))

    gt_static_explicit_indices_set = set(gt_indices_dict_for_target_interp.get('gt_static_indices', []))
    gt_unlabeled_indices_set = set(gt_indices_dict_for_target_interp.get('unlabeled_indices', []))
    # Combined set for what's considered "ground truth static" for FP calculation
    gt_effectively_static_indices_set = gt_static_explicit_indices_set.union(gt_unlabeled_indices_set)

    fp_indices_current_run = set() # To store FPs found by this cell's logic
    fn_indices_current_run = set() # To store FNs found by this cell's logic

    # Iterate through all points that were processed to determine current FPs/FNs
    # based on the "with_interp" trace data which includes the refinement.
    all_processed_indices_with_interp = debug_data_store.get("with_interp", {}).keys()

    fp_indices_current_run = set() # To store FPs found by this cell's logic
    fn_indices_current_run = set() # To store FNs found by this cell's logic

    current_fp_details = []
    current_fn_details = []
    
    # --- FP Analysis (Why MCC Refinement Failed) ---
    print("\n--- Analyzing Current False Positives (FPs) after MCC Refinement ---")
    fp_mcc_refinement_analysis = []

    for pt_idx in all_processed_indices_with_interp:
        data_w = debug_data_store.get("with_interp", {}).get(pt_idx)
        if not data_w:
            continue

        # Determine if this point is an FP in the current run (with refinement)
        is_fp = False
        if pt_idx in gt_effectively_static_indices_set and data_w.get("final_label") == "OCCLUDING_IMAGE":
            is_fp = True
            fp_indices_current_run.add(pt_idx)
            current_fp_details.append({
                "pt_idx": pt_idx, 
                "final_label_w": data_w.get("final_label"),
                "gt_is_static": True
            })


        if is_fp:
            # This point is an FP even after the MCC refinement step (or if MCC refinement wasn't applicable)
            analysis_entry = {
                "pt_idx": pt_idx,
                "T4_perp_event_passed_w": data_w.get("T4_perp_event_passed"),
                "preliminary_is_dynamic_w": data_w.get("preliminary_is_dynamic_before_refinement"),
                "mcc_refinement_performed_w": data_w.get("MCC_Refinement_performed"),
                "mcc_refinement_result_bool_w": data_w.get("MCC_Refinement_result_bool"), # Should be False for FPs
                "mcc_refinement_reason_w": "N/A",
                "mcc_refinement_interp_attempted_w": "N/A",
                "mcc_refinement_interp_depth_w": "N/A",
                "mcc_refinement_direct_comp_attempted_w": "N/A",
            }

            if data_w.get("MCC_Refinement_performed"): # If refinement was done
                if data_w.get("MCC_Refinement_result_bool") == False: # And it failed to make it static
                    refinement_debug = data_w.get("MCC_Refinement_full_debug", {})
                    analysis_entry["mcc_refinement_reason_w"] = refinement_debug.get("reason_for_result", "No debug reason")
                    
                    # Check details of the first relevant DI in refinement MCC if available
                    relevant_dis_details = refinement_debug.get('relevant_dis_details', [])
                    if relevant_dis_details:
                        first_di_detail = relevant_dis_details[0] # Examine the first one for a hint
                        analysis_entry["mcc_refinement_interp_attempted_w"] = first_di_detail.get('attempted_interpolation')
                        analysis_entry["mcc_refinement_interp_depth_w"] = first_di_detail.get('interpolation_result_depth')
                        analysis_entry["mcc_refinement_direct_comp_attempted_w"] = first_di_detail.get('attempted_direct_comparison')
                else:
                    # This case should ideally not happen for an FP if refinement was performed and succeeded
                    analysis_entry["mcc_refinement_reason_w"] = "MCC Refinement was True, but point still FP - check logic!" 
            else: # Refinement was not performed (e.g. preliminary was not dynamic, or MCC disabled)
                 analysis_entry["mcc_refinement_reason_w"] = "MCC Refinement not performed for this FP."


            fp_mcc_refinement_analysis.append(analysis_entry)

    print(f"Total FPs in current run (with MCC refinement): {len(fp_indices_current_run)}")
    if fp_mcc_refinement_analysis:
        df_fp_refinement = pd.DataFrame(fp_mcc_refinement_analysis)
        print("Snapshot of FPs and why MCC Refinement might have failed (showing first 5):")
        with pd.option_context('display.max_columns', None, 'display.width', 1500):
            print(df_fp_refinement.head())
        
        # Count common reasons for MCC refinement failure
        if not df_fp_refinement.empty:
            print("\nCounts of MCC Refinement Failure Reasons for FPs (based on first DI checked in MCC):")
            # Filter for FPs where refinement was performed but failed
            failed_refinement_fps = df_fp_refinement[
                (df_fp_refinement["mcc_refinement_performed_w"] == True) & \
                (df_fp_refinement["mcc_refinement_result_bool_w"] == False)
            ]
            if not failed_refinement_fps.empty:
                print(f"  Total FPs where MCC refinement was performed but failed: {len(failed_refinement_fps)}")
                
                common_reasons = failed_refinement_fps["mcc_refinement_reason_w"].value_counts()
                print("  Common 'reason_for_result' from MCC_Refinement_full_debug:")
                print(common_reasons.head(10))

                interp_failed_no_surface = failed_refinement_fps[
                    (failed_refinement_fps["mcc_refinement_interp_attempted_w"] == True) & \
                    (failed_refinement_fps["mcc_refinement_interp_depth_w"].isnull())
                ]
                print(f"\n  FPs where interp attempted, but no surface found (depth is None): {len(interp_failed_no_surface)}")
                
                interp_found_surface_not_consistent = failed_refinement_fps[
                     (failed_refinement_fps["mcc_refinement_interp_attempted_w"] == True) & \
                     (failed_refinement_fps["mcc_refinement_interp_depth_w"].notnull()) 
                     # Implies depth consistent check failed, as mcc_refinement_result_bool_w is False
                ]
                # This check is a bit indirect, relies on the overall MCC result being False
                # A more precise check would be if 'match_found_in_di' was False for interp in the debug details.
                print(f"  FPs where interp attempted, found surface, but likely not depth-consistent with target: {len(interp_found_surface_not_consistent)}")
                # For these, you'd look at the actual target depth vs interpolated depth and epsilons.

            else:
                print("  No FPs found where MCC refinement was performed and failed.")
    else:
        print("No FPs found in the current run to analyze for MCC refinement failure.")


    # --- FN Categorization ---
    print("\n\n--- Categorizing Current False Negatives (FNs) ---")
    fn_by_label_count = {"OCCLUDED_BY_IMAGE": 0, "UNDETERMINED": 0, "OTHER_STATIC_FN":0}
    fns_occluded_by_image_details = []
    fns_undetermined_details = []

    for pt_idx in all_processed_indices_with_interp:
        data_w = debug_data_store.get("with_interp", {}).get(pt_idx)
        if not data_w:
            continue
        
        is_fn = False
        if pt_idx in gt_dynamic_indices_set:
            final_label_w_str = data_w.get("final_label")
            if final_label_w_str == "OCCLUDED_BY_IMAGE":
                is_fn = True
                fn_by_label_count["OCCLUDED_BY_IMAGE"] += 1
                fns_occluded_by_image_details.append({
                    "pt_idx": pt_idx,
                    "T1_outcome_after_mcc_w": data_w.get("T1_outcome_after_mcc"),
                    "T1_mcc_result_bool_w": data_w.get("T1_mcc_result_bool"),
                    "MCC_Refinement_overrode_to_static_w": data_w.get("MCC_Refinement_overrode_to_static"),
                    "MCC_Refinement_result_bool_w": data_w.get("MCC_Refinement_result_bool")
                    # Add T1_mcc_full_debug and MCC_Refinement_full_debug reasons if needed
                })
            elif final_label_w_str == "UNDETERMINED":
                is_fn = True
                fn_by_label_count["UNDETERMINED"] += 1
                fns_undetermined_details.append({
                    "pt_idx": pt_idx,
                    "OUTCOME_T1_T4_w": data_w.get("OUTCOME_T1_T4"),
                    "OUTCOME_T2_w": data_w.get("OUTCOME_T2"),
                    "OUTCOME_T3_w": data_w.get("OUTCOME_T3"),
                    "T4_perp_event_passed_w": data_w.get("T4_perp_event_passed"),
                    "Test2_passed_w": data_w.get("Test2_passed"),
                    "Test3_chain_passed_w": data_w.get("Test3_chain_passed"),
                    "T3_mcc_rejects_P_w": data_w.get("T3_mcc_rejects_P")
                })
            elif final_label_w_str in ["EMPTY_IN_IMAGE", "PRELABELED_STATIC_GROUND"]: # Other static-like labels
                is_fn = True
                fn_by_label_count["OTHER_STATIC_FN"] += 1
        
        if is_fn:
            fn_indices_current_run.add(pt_idx)


    print(f"Total FNs in current run (with MCC refinement): {len(fn_indices_current_run)}")
    print(f"  FNs labeled 'OCCLUDED_BY_IMAGE': {fn_by_label_count['OCCLUDED_BY_IMAGE']}")
    print(f"  FNs labeled 'UNDETERMINED': {fn_by_label_count['UNDETERMINED']}")
    if fn_by_label_count['OTHER_STATIC_FN'] > 0:
        print(f"  FNs labeled with other static-like labels: {fn_by_label_count['OTHER_STATIC_FN']}")

    if fns_occluded_by_image_details:
        print("\nSnapshot of FNs mislabeled as OCCLUDED_BY_IMAGE (first 3):")
        df_fn_occ = pd.DataFrame(fns_occluded_by_image_details)
        with pd.option_context('display.max_columns', None, 'display.width', 1500):
            print(df_fn_occ.head(3))
        # Here you would further inspect the MCC reasons for these specific points
        # by looking up their T1_mcc_full_debug or MCC_Refinement_full_debug.

    if fns_undetermined_details:
        print("\nSnapshot of FNs mislabeled as UNDETERMINED (first 3):")
        df_fn_undet = pd.DataFrame(fns_undetermined_details)
        with pd.option_context('display.max_columns', None, 'display.width', 1500):
            print(df_fn_undet.head(3))
        # Here you would inspect why T1/T4, T2, and T3 all failed to label these as dynamic.


# %% Cell 3.9: Deep Dive into Test 3 Failures for UNDETERMINED FNs (Revised)

import pandas as pd
import numpy as np

if 'debug_data_store' not in locals() or \
   'gt_indices_dict_for_target_interp' not in locals() or \
   'fn_indices_current_run' not in locals():
    print("ERROR: Prerequisite data not found.")
else:
    print("\n--- Deep Dive into Test 3 Failures for UNDETERMINED False Negatives (FNs) ---")
    
    traced_data_with_interp = debug_data_store.get("with_interp", {})
    
    test3_failed_fns_details = []
    count_fns_test3_ran_but_failed = 0
    count_fns_test3_mcc_rejected = 0 # Still relevant from process_and_label_di's trace
    count_fns_test3_other_reason = 0

    for pt_idx in fn_indices_current_run:
        data_w = traced_data_with_interp.get(pt_idx)
        if not data_w:
            continue
        if data_w.get("final_label") != "UNDETERMINED":
            continue

        t3_mcc_rejects_p_initial = data_w.get("T3_mcc_rejects_P") # From process_and_label_di
        t3_chain_passed_final = data_w.get("Test3_chain_passed") # Overall result from process_and_label_di
        
        # Test3_full_debug is now expected to be a dictionary
        t3_execution_debug_dict = data_w.get("Test3_full_debug") 

        detail = {
            "pt_idx": pt_idx,
            "T3_initial_MCC_rejected_P": t3_mcc_rejects_p_initial, # From process_and_label_di
            "Test3_chain_overall_passed": t3_chain_passed_final, # From process_and_label_di
            "T3_execution_debug_available": isinstance(t3_execution_debug_dict, dict),
            "T3_execution_reason_for_fail": "N/A",
            "T3_execution_steps_details": None # To store the list of steps
        }

        if t3_mcc_rejects_p_initial == True:
            count_fns_test3_mcc_rejected += 1
            mcc_on_p_debug = data_w.get("T3_mcc_on_P_full_debug", {})
            detail["T3_execution_reason_for_fail"] = f"Initial MCC on P found it static. MCC Reason: {mcc_on_p_debug.get('reason_for_result', 'No MCC debug reason')}"
        elif t3_mcc_rejects_p_initial == False:
            # Test 3 chain logic was invoked
            if isinstance(t3_execution_debug_dict, dict):
                detail["T3_execution_reason_for_fail"] = t3_execution_debug_dict.get("reason_for_fail", "No specific reason in T3 execution debug.")
                detail["T3_execution_steps_details"] = t3_execution_debug_dict.get("steps_debug", []) # Get the list of steps
                
                if t3_execution_debug_dict.get("test_passed") == False: # Check the flag from execute_test3
                    count_fns_test3_ran_but_failed += 1
                elif t3_execution_debug_dict.get("test_passed") == True:
                     # This means execute_test3 returned True, but process_and_label_di's Test3_chain_passed might be False
                     # or the final label is still UNDETERMINED due to other reasons.
                     # This indicates a potential discrepancy or that T1/4 or T2 also didn't fire.
                    detail["T3_execution_reason_for_fail"] = "execute_test3 reported PASS, but FN is UNDETERMINED. Check overall logic."
                    count_fns_test3_other_reason +=1
                else: # test_passed flag missing or None
                    detail["T3_execution_reason_for_fail"] = "Test3 execution debug present, but 'test_passed' flag unclear."
                    count_fns_test3_other_reason +=1
            else:
                # Test3_full_debug was not a dictionary, meaning execute_test3 might not have returned detailed debug
                detail["T3_execution_reason_for_fail"] = "Test3 chain ran (initial MCC false), but no detailed execution debug found."
                if t3_chain_passed_final == False: # Rely on the flag from process_and_label_di
                    count_fns_test3_ran_but_failed += 1
                else:
                    count_fns_test3_other_reason +=1
        else: # t3_mcc_rejects_p_initial is None or other
            detail["T3_execution_reason_for_fail"] = f"Test3 initial MCC outcome unclear (value: {t3_mcc_rejects_p_initial})."
            count_fns_test3_other_reason +=1
            
        test3_failed_fns_details.append(detail)

    print(f"\nTotal UNDETERMINED FNs analyzed for Test 3 behavior: {len(test3_failed_fns_details)}")
    print(f"  FNs where Test 3 chain was REJECTED by initial MCC on P (from process_and_label_di): {count_fns_test3_mcc_rejected}")
    print(f"  FNs where Test 3 chain RAN but FAILED (based on execute_test3_perpendicular_motion debug): {count_fns_test3_ran_but_failed}")
    if count_fns_test3_other_reason > 0:
        print(f"  FNs with other/unclear Test 3 outcomes for UNDETERMINED FNs: {count_fns_test3_other_reason}")

    if test3_failed_fns_details:
        df_test3_fails = pd.DataFrame(test3_failed_fns_details)
        
        df_t3_ran_failed = df_test3_fails[
            (df_test3_fails["T3_initial_MCC_rejected_P"] == False) & \
            # Check the 'test_passed' flag from the execution debug if available
            # This requires Test3_full_debug to be a dict with 'test_passed'
            (df_test3_fails["T3_execution_debug_available"] == True) & \
            (df_test3_fails.apply(lambda row: data_w.get("Test3_full_debug", {}).get("test_passed") == False if isinstance(data_w.get("Test3_full_debug"), dict) else True, axis=1))
            # A simpler check if relying on Test3_chain_overall_passed from process_and_label_di
            # (df_test3_fails["Test3_chain_overall_passed"] == False)
        ]

        if not df_t3_ran_failed.empty:
            print(f"\nSnapshot of UNDETERMINED FNs where Test 3 chain ran but FAILED (first 5 or all):")
            # Select columns to display to avoid excessive width if steps_details is long
            cols_to_show = ["pt_idx", "T3_initial_MCC_rejected_P", "Test3_chain_overall_passed", "T3_execution_reason_for_fail"]
            with pd.option_context('display.max_colwidth', 200, 'display.max_columns', None, 'display.width', 2000):
                print(df_t3_ran_failed[cols_to_show].head(min(5, len(df_t3_ran_failed))))
            
            print("\nCommon reasons for Test 3 chain failure (from T3_execution_reason_for_fail):")
            if "T3_execution_reason_for_fail" in df_t3_ran_failed.columns:
                print(df_t3_ran_failed["T3_execution_reason_for_fail"].value_counts().head(10))

        # You can also print details for specific points and inspect their "T3_execution_steps_details"
        # For example, for the first point in df_t3_ran_failed:
        if not df_t3_ran_failed.empty:
            print(f"\nSnapshot of UNDETERMINED FNs where Test 3 chain ran but FAILED (first 5 or all):")
            cols_to_show = ["pt_idx", "T3_initial_MCC_rejected_P", "Test3_chain_overall_passed", "T3_execution_reason_for_fail"]
            with pd.option_context('display.max_colwidth', 200, 'display.max_columns', None, 'display.width', 2000):
                print(df_t3_ran_failed[cols_to_show].head(min(5, len(df_t3_ran_failed))))

            print("\nCommon reasons for Test 3 chain failure (from T3_execution_reason_for_fail):")
            if "T3_execution_reason_for_fail" in df_t3_ran_failed.columns:
                print(df_t3_ran_failed["T3_execution_reason_for_fail"].value_counts().head(10))

            # ADD THIS PART TO INSPECT STEP DETAILS for "No point found occluded"
            print("\nInspecting first FN where 'No point found occluded':")
            first_no_occluded_fn = df_t3_ran_failed[df_t3_ran_failed["T3_execution_reason_for_fail"] == "Failed: No point found in historical DI pixel that was occluded by current chain point."].head(1)
            if not first_no_occluded_fn.empty:
                pt_idx_inspect = first_no_occluded_fn.iloc[0]["pt_idx"]
                data_w_inspect = traced_data_with_interp.get(pt_idx_inspect)
                if data_w_inspect:
                    t3_exec_debug_inspect = data_w_inspect.get("Test3_full_debug")
                    if isinstance(t3_exec_debug_inspect, dict) and "steps_debug" in t3_exec_debug_inspect:
                        print(f"  Detailed steps for pt_idx: {pt_idx_inspect} (Reason: {t3_exec_debug_inspect.get('reason_for_fail')})")
                        for step_detail in t3_exec_debug_inspect["steps_debug"]:
                            print(f"    Step {step_detail.get('step')}: Status: {step_detail.get('status')}")
                            if step_detail.get('status') != "Success: Valid occluded point found, chain continues.":
                                print(f"      Occluder Global: {step_detail.get('point_that_occludes_global')}")
                                print(f"      Historical DI Idx: {step_detail.get('historical_di_checked_idx_in_lib')}, Timestamp: {step_detail.get('historical_di_checked_timestamp')}")
                                print(f"      Occluder Sph in Hist: {step_detail.get('sph_coords_of_occluder_in_hist_frame')}")
                                if step_detail.get("occlusion_check_details"):
                                    print("      Candidate Checks:")
                                    for cand_check in step_detail.get("occlusion_check_details", []):
                                        print(f"        HistCand Idx: {cand_check.get('hist_candidate_original_idx')}, Occluder Depth in Hist: {cand_check.get('depth_occluder_in_hist_frame')}, Cand Depth in Hist: {cand_check.get('depth_hist_cand_native_to_hist_frame')}, In Front: {cand_check.get('is_current_occluder_in_front_of_hist_cand')}")
                            if step_detail.get('status') != "Success: Valid occluded point found, chain continues.": # Stop after first failing step for brevity
                                break 
            else:
                print("Could not find an example FN for 'No point found occluded' to inspect.")
        else:
            print("No UNDETERMINED FNs found to analyze for Test 3 behavior, or no trace data for them.")

# %% Cell 3.10: Deep Dive into T3 Initial MCC Rejections for FNs

import pandas as pd
import numpy as np

if 'debug_data_store' not in locals() or \
   'fn_indices_current_run' not in locals() or \
   'df_test3_fails' not in locals(): # df_test3_fails from Cell 3.9 might be useful
    print("ERROR: Prerequisite data (debug_data_store, fn_indices_current_run, df_test3_fails) not found.")
    print("Please ensure Cell 3.5, 3.8, and 3.9 have run successfully.")
else:
    print("\n--- Deep Dive into Test 3 Initial MCC Rejections for UNDETERMINED FNs ---")
    
    traced_data_with_interp = debug_data_store.get("with_interp", {})
    
    # Filter for FNs where T3 was rejected by the initial MCC on P
    # Using df_test3_fails from Cell 3.9 if available and correctly populated
    if 'df_test3_fails' in locals() and not df_test3_fails.empty:
        fns_t3_mcc_rejected_df = df_test3_fails[df_test3_fails["T3_initial_MCC_rejected_P"] == True]
        pt_indices_to_inspect = fns_t3_mcc_rejected_df["pt_idx"].tolist()
        print(f"Found {len(pt_indices_to_inspect)} FNs where T3 initial MCC rejected P (from Cell 3.9's dataframe).")
    else:
        # Fallback: Manually filter from fn_indices_current_run and traced_data_with_interp
        print("Warning: df_test3_fails not available or empty. Manually filtering FNs for T3 MCC rejection.")
        pt_indices_to_inspect = []
        for pt_idx in fn_indices_current_run:
            data_w = traced_data_with_interp.get(pt_idx)
            if data_w and data_w.get("final_label") == "UNDETERMINED" and data_w.get("T3_mcc_rejects_P") == True:
                pt_indices_to_inspect.append(pt_idx)
        print(f"Found {len(pt_indices_to_inspect)} FNs (manual filter) where T3 initial MCC rejected P.")

    if not pt_indices_to_inspect:
        print("No FNs found where Test 3 was rejected by initial MCC to analyze.")
    else:
        # Let's inspect the first few (e.g., up to 3)
        for i, pt_idx in enumerate(pt_indices_to_inspect[:min(3, len(pt_indices_to_inspect))]):
            data_w = traced_data_with_interp.get(pt_idx)
            if not data_w:
                print(f"\n--- FN pt_idx: {pt_idx} ---")
                print("  No trace data found in debug_data_store['with_interp'].")
                continue

            t3_mcc_on_p_debug = data_w.get("T3_mcc_on_P_full_debug")

            print(f"\n--- FN pt_idx: {pt_idx} (T3 Initial MCC Rejected P) ---")
            if not isinstance(t3_mcc_on_p_debug, dict):
                print(f"  T3_mcc_on_P_full_debug is not a dictionary or is missing. Value: {t3_mcc_on_p_debug}")
                continue

            print(f"  MCC Overall Result: {t3_mcc_on_p_debug.get('map_consistent_result')}")
            print(f"  MCC Reason: {t3_mcc_on_p_debug.get('reason_for_result')}")
            print(f"  Config - Interp Enabled during MCC call: {data_w.get('config_interp_enabled_during_call')}") # From process_and_label_di trace
            print(f"  Config - MCC Interp Enabled (param): {t3_mcc_on_p_debug.get('config_mc_interp_enabled')}")
            print(f"  Config - MCC Threshold Mode: {t3_mcc_on_p_debug.get('config_mc_threshold_mode')}, Count: {t3_mcc_on_p_debug.get('config_mc_threshold_value_count')}")
            print(f"  Matches Found: {t3_mcc_on_p_debug.get('consistent_matches_final_count')} / {t3_mcc_on_p_debug.get('num_dis_actually_checked_final')} DIs checked (Total relevant: {t3_mcc_on_p_debug.get('relevant_dis_count')})")
            
            print("\n  Details per Relevant Historical DI:")
            relevant_dis_details = t3_mcc_on_p_debug.get('relevant_dis_details', [])
            if not relevant_dis_details:
                print("    No relevant DI details found in T3_mcc_on_P_full_debug.")
            
            for di_detail_idx, di_detail in enumerate(relevant_dis_details):
                print(f"    --- Hist DI {di_detail_idx} (TS: {di_detail.get('di_timestamp'):.2f}) ---")
                print(f"      Projected Successfully: {di_detail.get('projection_successful')}")
                if not di_detail.get('projection_successful'):
                    print(f"        Reason Skipped: {di_detail.get('reason_skipped_di_entirely')}")
                    continue # Skip to next DI detail if projection failed

                print(f"      Match Found in this DI: {di_detail.get('match_found_in_di')}")
                
                # Interpolation Details
                if di_detail.get('attempted_interpolation'):
                    print("      Interpolation Attempted:")
                    interp_surf_info = di_detail.get('interp_depth_target_vs_surface', {})
                    print(f"        Interpolated Surface Depth: {interp_surf_info.get('interpolated_surface_depth')}")
                    print(f"        Target Depth in Hist DI: {interp_surf_info.get('target_depth_in_hist_di')}")
                    print(f"        Was Consistent (Interp): {interp_surf_info.get('was_consistent')}")
                    if interp_surf_info.get('interpolated_surface_depth') is not None:
                         print(f"        Bounds: [{interp_surf_info.get('lower_bound')}, {interp_surf_info.get('upper_bound')}]")
                else:
                    print("      Interpolation NOT Attempted (or not applicable for this DI).")

                # Direct Comparison Details (if performed and relevant)
                if di_detail.get('attempted_direct_comparison'):
                    print("      Direct Comparison Attempted:")
                    print(f"        Pixel Had Content: {di_detail.get('direct_comparison_pixel_had_content')}")
                    if di_detail.get('direct_comparison_pixel_had_content'):
                        print("        Static Points Checked in Pixel:")
                        static_pts_checked = di_detail.get('static_points_in_pixel_details', [])
                        if not static_pts_checked:
                            print("          No static points detailed (or none met criteria to be checked).")
                        for static_pt_idx, static_pt_detail in enumerate(static_pts_checked):
                            if static_pt_detail.get('match_found'): # Only print details for actual matches or key failures
                                print(f"          - Candidate {static_pt_idx} (Orig Idx: {static_pt_detail.get('original_index_in_di')}): MATCHED")
                                print(f"            Label: {static_pt_detail.get('label_in_di')}, IsStatic: {static_pt_detail.get('is_static_label')}")
                                print(f"            Depths (Target vs StaticCand): {static_pt_detail.get('depth_target')} vs {static_pt_detail.get('depth_static_cand')}, Consistent: {static_pt_detail.get('depth_consistent')}")
                                print(f"            Phi Consistent: {static_pt_detail.get('phi_consistent')}, Theta Consistent: {static_pt_detail.get('theta_consistent')}")
                                break # Show first match and stop for brevity for this DI
                        # If no match was found but direct comparison was attempted on contentful pixel,
                        # you might want to log the first non-matching candidate's details.
                # elif not found_consistent_static_point_in_this_di and not di_detail.get('attempted_interpolation'): # Typo: found_consistent_static_point_in_this_di is not defined here
                #     # Rephrase: if no match yet and direct comparison was not even attempted
                #      pass # Covered by other flags

            if i >= 2: # Limit to inspecting first 3 FNs in detail
                break

# %% Cell 3.11: Calculate Overall Classification Metrics (Revised to Treat GT Unlabeled as Static)

import numpy as np
# OcclusionResult is imported in Cell 0

try:
    MDET_DYNAMIC_LABEL = OcclusionResult.OCCLUDING_IMAGE.value
except NameError:
    print("Warning: OcclusionResult not defined in the current scope of Cell 3.11.")
    print("Please ensure Cell 0 (Setup and Imports) has been run.")
    print("Assuming MDET_DYNAMIC_LABEL = 0 for OCCLUDING_IMAGE as a fallback.")
    MDET_DYNAMIC_LABEL = 0

if 'gt_indices_dict_for_target_interp' not in locals() or gt_indices_dict_for_target_interp is None:
    print("Error: Ground truth indices ('gt_indices_dict_for_target_interp') not found or is None.")
elif 'labels_with_interp_final' not in locals():
    print("Error: MDetector's final labels ('labels_with_interp_final') not found.")
elif 'num_pts_in_target_di' not in locals():
    print("Error: Total number of points in target DI ('num_pts_in_target_di') not found.")
else:
    predicted_labels = labels_with_interp_final

    # --- Construct Ground Truth Binary Labels ---
    # Initialize GT as all static (0). This will inherently include GT_Static and GT_Unlabeled.
    gt_binary = np.zeros(num_pts_in_target_di, dtype=int)
    
    # Mark GT dynamic points as 1
    gt_dynamic_indices = gt_indices_dict_for_target_interp.get('gt_dynamic_indices', np.array([], dtype=int))
    if gt_dynamic_indices.size > 0:
        # Ensure indices are within bounds, though they should be if generated correctly
        valid_dynamic_indices = gt_dynamic_indices[gt_dynamic_indices < num_pts_in_target_di]
        gt_binary[valid_dynamic_indices] = 1
        if len(valid_dynamic_indices) != len(gt_dynamic_indices):
            print(f"Warning: Some GT dynamic indices were out of bounds for num_pts_in_target_di ({num_pts_in_target_di}).")

    # No explicit unlabeled_mask needed if unlabeled are treated as static (0 by default)
    num_actually_labeled_points = num_pts_in_target_di # All points are now considered "labeled" for metric purposes

    # --- Convert Predicted labels to binary: 1 for Dynamic, 0 for Static ---
    predicted_binary = (predicted_labels == MDET_DYNAMIC_LABEL).astype(int)

    # --- Calculate True Positives, False Positives, True Negatives, False Negatives ---
    tp = np.sum((gt_binary == 1) & (predicted_binary == 1))
    fp = np.sum((gt_binary == 0) & (predicted_binary == 1)) # GT is Static (or Unlabeled), Predicted is Dynamic
    tn = np.sum((gt_binary == 0) & (predicted_binary == 0)) # GT is Static (or Unlabeled), Predicted is Static
    fn = np.sum((gt_binary == 1) & (predicted_binary == 0)) # GT is Dynamic, Predicted is Static

    print(f"\n--- Overall Sweep Metrics (After MCC Refinement, GT Unlabeled treated as Static) ---")
    print(f"  Total points in DI: {num_pts_in_target_di}")
    gt_static_explicit_indices = gt_indices_dict_for_target_interp.get('gt_static_indices', np.array([], dtype=int))
    unlabeled_indices = gt_indices_dict_for_target_interp.get('unlabeled_indices', np.array([], dtype=int))
    print(f"  GT Dynamic points: {len(gt_dynamic_indices)}")
    print(f"  GT Explicitly Static points: {len(gt_static_explicit_indices)}")
    print(f"  GT Unlabeled points (treated as Static): {len(unlabeled_indices)}")
    print(f"  Total GT Static points for metrics (Explicit Static + Unlabeled): {np.sum(gt_binary == 0)}")
    print(f"  --------------------------------------------------------------------")
    print(f"  True Positives (TP)  | Dynamic correctly identified as Dynamic:      {tp}")
    print(f"  False Positives (FP) | Static/Unlabeled incorrectly as Dynamic:    {fp}")
    print(f"  True Negatives (TN)  | Static/Unlabeled correctly as Static:       {tn}")
    print(f"  False Negatives (FN) | Dynamic missed (as Static/Undetermined):    {fn}")
    print(f"  --------------------------------------------------------------------")

    # --- Calculate Metrics ---
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    accuracy = (tp + tn) / num_actually_labeled_points if num_actually_labeled_points > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print(f"  Precision:                            {precision:.4f}")
    print(f"  Recall:                               {recall:.4f}")
    print(f"  Accuracy:                             {accuracy:.4f}")
    print(f"  F1:                                   {f1_score:.4f}")
    print(f"  FPR:                                  {fpr:.4f}")
    print(f"  Specificity:                          {specificity:.4f}")
    print(f"  --------------------------------------------------------------------")

    print("\n  Sanity Check with Cell 3.5's 'Results WITH Interpolation (Controlled Run 1)' numbers:")
    print(f"    Calculated FP here: {fp} (Compare with 'Static -> Dynamic' FPs from Cell 3.5)")
    print(f"    Calculated FN here: {fn} (Compare with 'Dynamic -> Static/Undet' FNs from Cell 3.5)")
    if 'fp_with_interp_ctrl' in locals() and 'fn_with_interp_ctrl' in locals():
         print(f"    Cell 3.5 reported FP: {fp_with_interp_ctrl}, FN: {fn_with_interp_ctrl}")
         if fp == fp_with_interp_ctrl and fn == fn_with_interp_ctrl:
             print("    FP/FN counts match Cell 3.5's 'With Interpolation' controlled run. Metrics are consistent.")
         else:
             print("    WARNING: FP/FN counts DO NOT match Cell 3.5's 'With Interpolation' controlled run. Review which labels are being compared.")
    else:
        print("    Variables 'fp_with_interp_ctrl' or 'fn_with_interp_ctrl' from Cell 3.5 not found for direct comparison.")

# %% Cell 3.12: Comprehensive Post-Hoc Analysis of Adaptive Epsilon Strategies

import pandas as pd
import numpy as np
from typing import Optional, Dict, Any # Ensure Optional is imported

print("\n--- Cell 3.12: Comprehensive Post-Hoc Analysis of Adaptive Epsilon Strategies ---")

# --- 0. Ensure Prerequisites ---
if 'config_accessor' not in locals() or config_accessor is None:
    print("ERROR: config_accessor not found. Please run the configuration cell.")
    raise NameError("config_accessor not defined")
if 'debug_data_store' not in locals():
    print("ERROR: debug_data_store not found. Please run cells that populate it.")
    raise NameError("debug_data_store not defined")
if 'fp_indices_current_run' not in locals(): # Assuming Cell 3.8 defines this
    print("WARNING: fp_indices_current_run not found. Analysis for FPs will be skipped.")
    fp_indices_current_run = []
if 'fn_indices_current_run' not in locals(): # Assuming Cell 3.8 defines this
    print("WARNING: fn_indices_current_run not found. Analysis for FNs will be limited.")
    fn_indices_current_run = []
if 'df_test3_fails' not in locals(): # Assuming Cell 3.9 defines this
     print("WARNING: df_test3_fails not found. Test 3 specific FN analysis might be incomplete.")
     df_test3_fails = pd.DataFrame() # Empty DataFrame to avoid errors

traced_data_with_interp = debug_data_store.get("with_interp", {})

# --- 1. Load ALL Adaptive Epsilon Simulation Parameters from Config ---

# For Test 3 (epsilon_depth_occlusion)
# Note: Your existing cell hardcoded these, let's try to use the config structure we defined
# This assumes 'adaptive_epsilon_depth_config' is under 'occlusion_determination'
# and 'adaptive_epsilon_simulation_params' was a temporary name for Test 3 specific simulation.
# We'll use the new structure for consistency.
occlusion_params_cfg = config_accessor.get_occlusion_determination_params()
adaptive_cfg_occ_depth = occlusion_params_cfg.get('adaptive_epsilon_depth_config', {})
base_eps_occ_depth = occlusion_params_cfg.get('epsilon_depth', 0.5) # Default if not in config

sim_enabled_occ = adaptive_cfg_occ_depth.get('enabled', False)
sim_dthr_occ = adaptive_cfg_occ_depth.get('dthr', 30.0)
sim_kthr_occ = adaptive_cfg_occ_depth.get('kthr', 0.05) # Defaulting to positive kthr for general case
sim_dmax_occ = adaptive_cfg_occ_depth.get('dmax', 2.0)
sim_dmin_occ = adaptive_cfg_occ_depth.get('dmin', 0.1)

# For MCC (epsilon_depth_forward_map & epsilon_depth_backward_map)
mc_params_cfg = config_accessor.get_map_consistency_params()
adaptive_cfg_mc_fwd = mc_params_cfg.get('adaptive_epsilon_forward_config', {})
base_eps_mc_fwd = mc_params_cfg.get('epsilon_depth_forward_m', 1.0)

sim_enabled_mc_fwd = adaptive_cfg_mc_fwd.get('enabled', False)
sim_dthr_mc_fwd = adaptive_cfg_mc_fwd.get('dthr', 20.0)
sim_kthr_mc_fwd = adaptive_cfg_mc_fwd.get('kthr', 0.07)
sim_dmax_mc_fwd = adaptive_cfg_mc_fwd.get('dmax', 3.0)
sim_dmin_mc_fwd = adaptive_cfg_mc_fwd.get('dmin', 0.2)

adaptive_cfg_mc_bwd = mc_params_cfg.get('adaptive_epsilon_backward_config', {})
base_eps_mc_bwd = mc_params_cfg.get('epsilon_depth_backward_m', 0.3)

sim_enabled_mc_bwd = adaptive_cfg_mc_bwd.get('enabled', False)
sim_dthr_mc_bwd = adaptive_cfg_mc_bwd.get('dthr', 20.0)
sim_kthr_mc_bwd = adaptive_cfg_mc_bwd.get('kthr', 0.03)
sim_dmax_mc_bwd = adaptive_cfg_mc_bwd.get('dmax', 1.0)
sim_dmin_mc_bwd = adaptive_cfg_mc_bwd.get('dmin', 0.05)

print("Adaptive Epsilon Simulation Settings Loaded:")
print(f"  Occlusion Depth: enabled={sim_enabled_occ}, base_eps={base_eps_occ_depth:.2f}, dthr={sim_dthr_occ}, kthr={sim_kthr_occ}, dmax={sim_dmax_occ}, dmin={sim_dmin_occ}")
print(f"  MCC Forward:     enabled={sim_enabled_mc_fwd}, base_eps={base_eps_mc_fwd:.2f}, dthr={sim_dthr_mc_fwd}, kthr={sim_kthr_mc_fwd}, dmax={sim_dmax_mc_fwd}, dmin={sim_dmin_mc_fwd}")
print(f"  MCC Backward:    enabled={sim_enabled_mc_bwd}, base_eps={base_eps_mc_bwd:.2f}, dthr={sim_dthr_mc_bwd}, kthr={sim_kthr_mc_bwd}, dmax={sim_dmax_mc_bwd}, dmin={sim_dmin_mc_bwd}")

# --- 2. Define calculate_adaptive_epsilon locally (consistent with adaptive_epsilon_utils.py) ---
def calculate_adaptive_epsilon_local(
    d_anchor: Optional[float],
    di_base: float,
    dthr: float, kthr: float, dmax: float, dmin: float, # Pass individual params
    is_enabled: bool = True # Allow disabling per call if needed
) -> float:
    if not is_enabled or d_anchor is None or d_anchor < 0:
        return di_base

    adjustment = 0.0
    # Logic for kthr >= 0 (increasing or flat)
    if kthr >= 0:
        if d_anchor > dthr:
            adjustment = kthr * (d_anchor - dthr)
        calculated_epsilon = di_base + adjustment
    # Logic for kthr < 0 (decreasing or flat)
    else: # kthr < 0
        if d_anchor > dthr: # Only decrease if beyond dthr
            adjustment = kthr * (d_anchor - dthr) # adjustment will be negative
            calculated_epsilon = di_base + adjustment
        else: # d_anchor <= dthr, kthr is negative, so no change from base
            calculated_epsilon = di_base
            
    final_epsilon = np.clip(calculated_epsilon, dmin, dmax)
    return float(final_epsilon)

# --- Helper for printing diagnostics (limited output) ---
MAX_DIAGNOSTICS_PER_SECTION = 3
diag_counters = {"test3_fn": 0, "mcc_fp": 0, "mcc_fn_t1": 0, "mcc_fn_t3":0}

def print_diag(section_key, *args):
    if diag_counters[section_key] < MAX_DIAGNOSTICS_PER_SECTION:
        print("    ", *args)
        # diag_counters[section_key] += 1 # Increment outside if a block is printed

# =====================================================================================
# --- SECTION A: Test 3 "No Point Found Occluded" FNs (epsilon_depth_occlusion) ---
# =====================================================================================
print("\n\n--- SECTION A: Test 3 'No Point Found Occluded' FNs ---")
if not sim_enabled_occ:
    print("Adaptive epsilon for Occlusion Depth (Test 3) is disabled. Skipping Section A.")
else:
    target_fn_pt_indices_for_t3_sim = []
    if not df_test3_fails.empty:
        target_failure_reason_t3 = "Failed: No point found in historical DI pixel that was occluded by current chain point."
        fns_for_t3_sim_df = df_test3_fails[
            (df_test3_fails["T3_initial_MCC_rejected_P"] == False) &
            (df_test3_fails["T3_execution_reason_for_fail"] == target_failure_reason_t3)
        ]
        target_fn_pt_indices_for_t3_sim = fns_for_t3_sim_df["pt_idx"].tolist()
    print(f"Found {len(target_fn_pt_indices_for_t3_sim)} FNs where Test 3 failed due to '{target_failure_reason_t3}'.")

    if not target_fn_pt_indices_for_t3_sim:
        print("No FNs matching for Test 3 simulation.")
    else:
        resolved_t3_fns_count = 0
        resolved_t3_links_count = 0
        
        for pt_idx in target_fn_pt_indices_for_t3_sim:
            fn_data_w = traced_data_with_interp.get(pt_idx)
            if not fn_data_w: continue
            test3_full_debug = fn_data_w.get("Test3_full_debug")
            if not isinstance(test3_full_debug, dict): continue

            # di_base_from_trace_t3 = test3_full_debug.get("fixed_epsilon_depth_occlusion_used_in_logic", base_eps_occ_depth)
            # For simulation, always use the configured base_eps_occ_depth as di_base
            di_base_t3 = base_eps_occ_depth

            steps_debug_list = test3_full_debug.get("steps_debug", [])
            fn_resolved_this_time_t3 = False

            for step_info in steps_debug_list:
                if step_info.get("status") == target_failure_reason_t3:
                    d_anchor_occluder_t3 = step_info.get("d_anchor_of_occluder_for_this_step")
                    occlusion_candidates = step_info.get("occlusion_check_details", [])
                    
                    for cand_idx, cand_check in enumerate(occlusion_candidates):
                        if cand_check.get("forms_next_link_in_chain") == True: continue
                        
                        depth_o = cand_check.get("depth_occluder_in_hist_frame")
                        depth_h = cand_check.get("depth_hist_cand_native_to_hist_frame")

                        if depth_o is not None and depth_h is not None:
                            delta_actual = depth_h - depth_o
                            fixed_eps_used_t3 = cand_check.get("epsilon_used_for_check", di_base_t3)

                            if diag_counters["test3_fn"] < MAX_DIAGNOSTICS_PER_SECTION and delta_actual <= 0:
                                print_diag("test3_fn", f"Pt {pt_idx}, T3 Step {step_info.get('step')}, Cand {cand_idx}: delta_actual={delta_actual:.3f} <= 0. Geom fail.")
                                diag_counters["test3_fn"] += 1


                            if delta_actual > 0:
                                sim_adaptive_eps_t3 = calculate_adaptive_epsilon_local(
                                    d_anchor_occluder_t3, di_base_t3,
                                    sim_dthr_occ, sim_kthr_occ, sim_dmax_occ, sim_dmin_occ,
                                    sim_enabled_occ
                                )
                                would_pass_adaptive_t3 = depth_o < (depth_h - sim_adaptive_eps_t3)

                                if diag_counters["test3_fn"] < MAX_DIAGNOSTICS_PER_SECTION:
                                    print_diag("test3_fn", f"Pt {pt_idx}, T3 Step {step_info.get('step')}, Cand {cand_idx}:")
                                    print_diag("test3_fn", f"  d_anchor={d_anchor_occluder_t3:.2f}m, di_base={di_base_t3:.2f}m, delta_actual={delta_actual:.3f}m")
                                    print_diag("test3_fn", f"  FixedEps: {fixed_eps_used_t3:.3f}. SimAdaptiveEps: {sim_adaptive_eps_t3:.3f}")
                                    print_diag("test3_fn", f"  Condition: {depth_o:.3f} < ({depth_h:.3f} - SimEps:{sim_adaptive_eps_t3:.3f}) = {depth_h - sim_adaptive_eps_t3:.3f} -> {would_pass_adaptive_t3}")
                                    diag_counters["test3_fn"] += 1


                                if would_pass_adaptive_t3:
                                    resolved_t3_links_count += 1
                                    fn_resolved_this_time_t3 = True; break
                    if fn_resolved_this_time_t3: break
            if fn_resolved_this_time_t3: resolved_t3_fns_count += 1
        
        print(f"Test 3 FN Summary: Resolved Links: {resolved_t3_links_count}, Resolved FNs: {resolved_t3_fns_count}/{len(target_fn_pt_indices_for_t3_sim)}")

# =====================================================================================
# --- SECTION B: MCC for False Positives (MCC Refinement) ---
# =====================================================================================
print("\n\n--- SECTION B: MCC for False Positives (Refinement) ---")
if not (sim_enabled_mc_fwd or sim_enabled_mc_bwd): # Check if at least one is enabled
    print("Adaptive epsilon for MCC (Fwd/Bwd) is disabled. Skipping Section B.")
else:
    target_fp_indices_for_mcc_sim = fp_indices_current_run # Use all FPs
    print(f"Analyzing {len(target_fp_indices_for_mcc_sim)} FPs for MCC refinement changes.")

    resolved_fps_by_mcc_count = 0
    
    for pt_idx in target_fp_indices_for_mcc_sim:
        fp_data_w = traced_data_with_interp.get(pt_idx)
        if not fp_data_w: continue
        
        # Check if MCC refinement was performed and failed (i.e., point remained dynamic)
        if not fp_data_w.get("MCC_Refinement_performed", False) or fp_data_w.get("MCC_Refinement_overrode_to_static", False):
            continue # MCC didn't run or it already correctly labeled it static (not an FP cause by MCC refinement failure)

        mcc_refinement_debug = fp_data_w.get("MCC_Refinement_full_debug")
        if not isinstance(mcc_refinement_debug, dict): continue

        d_anchor_fp = mcc_refinement_debug.get("d_anchor_of_point_global_for_mcc")
        # base_eps_fwd_trace = mcc_refinement_debug.get("config_epsilon_depth_forward_map", base_eps_mc_fwd) # Get from trace if available
        # base_eps_bwd_trace = mcc_refinement_debug.get("config_epsilon_depth_backward_map", base_eps_mc_bwd)
        # For simulation, always use the configured base epsilon
        di_base_mc_fwd = base_eps_mc_fwd
        di_base_mc_bwd = base_eps_mc_bwd


        fp_resolved_this_time = False
        for di_detail in mcc_refinement_debug.get("relevant_dis_details", []):
            # We are interested if any DI check that FAILED with fixed epsilon would PASS with adaptive
            if di_detail.get("match_found_in_di", False): continue # Already matched in this DI

            # Check Interpolation Path
            if di_detail.get("attempted_interpolation", False) and isinstance(di_detail.get("interp_depth_target_vs_surface"), dict):
                interp_info = di_detail["interp_depth_target_vs_surface"]
                target_d = interp_info.get("target_depth_in_hist_di")
                surface_d = interp_info.get("interpolated_surface_depth")
                
                if target_d is not None and surface_d is not None:
                    sim_eps_fwd = calculate_adaptive_epsilon_local(d_anchor_fp, di_base_mc_fwd, sim_dthr_mc_fwd, sim_kthr_mc_fwd, sim_dmax_mc_fwd, sim_dmin_mc_fwd, sim_enabled_mc_fwd)
                    sim_eps_bwd = calculate_adaptive_epsilon_local(d_anchor_fp, di_base_mc_bwd, sim_dthr_mc_bwd, sim_kthr_mc_bwd, sim_dmax_mc_bwd, sim_dmin_mc_bwd, sim_enabled_mc_bwd)
                    
                    consistent_adaptive = (surface_d - sim_eps_bwd <= target_d <= surface_d + sim_eps_fwd)
                    if consistent_adaptive:
                        fp_resolved_this_time = True
                        if diag_counters["mcc_fp"] < MAX_DIAGNOSTICS_PER_SECTION:
                            print_diag("mcc_fp", f"FP Pt {pt_idx} (Interp): d_anchor={d_anchor_fp:.2f}, target_d={target_d:.2f}, surface_d={surface_d:.2f}")
                            print_diag("mcc_fp", f"  SimEpsFwd={sim_eps_fwd:.3f}, SimEpsBwd={sim_eps_bwd:.3f}. OrigMatch={interp_info.get('was_consistent')}, AdaptiveMatch={consistent_adaptive}")
                            diag_counters["mcc_fp"] += 1
                        break 
            if fp_resolved_this_time: break

            # Check Direct Comparison Path (if interp didn't resolve)
            if di_detail.get("attempted_direct_comparison", False):
                for static_cand_detail in di_detail.get("static_points_in_pixel_details", []):
                    if static_cand_detail.get("match_found", False): continue # Already matched this candidate

                    target_d = static_cand_detail.get("depth_target")
                    static_d = static_cand_detail.get("depth_static_cand")
                    if target_d is not None and static_d is not None and static_cand_detail.get("is_static_label"):
                        sim_eps_fwd = calculate_adaptive_epsilon_local(d_anchor_fp, di_base_mc_fwd, sim_dthr_mc_fwd, sim_kthr_mc_fwd, sim_dmax_mc_fwd, sim_dmin_mc_fwd, sim_enabled_mc_fwd)
                        sim_eps_bwd = calculate_adaptive_epsilon_local(d_anchor_fp, di_base_mc_bwd, sim_dthr_mc_bwd, sim_kthr_mc_bwd, sim_dmax_mc_bwd, sim_dmin_mc_bwd, sim_enabled_mc_bwd)

                        consistent_adaptive = (static_d - sim_eps_bwd <= target_d <= static_d + sim_eps_fwd)
                        if consistent_adaptive:
                            fp_resolved_this_time = True
                            if diag_counters["mcc_fp"] < MAX_DIAGNOSTICS_PER_SECTION:
                                print_diag("mcc_fp", f"FP Pt {pt_idx} (Direct): d_anchor={d_anchor_fp:.2f}, target_d={target_d:.2f}, static_d={static_d:.2f}")
                                print_diag("mcc_fp", f"  SimEpsFwd={sim_eps_fwd:.3f}, SimEpsBwd={sim_eps_bwd:.3f}. OrigMatch=False, AdaptiveMatch={consistent_adaptive}")
                                diag_counters["mcc_fp"] += 1
                            break
                if fp_resolved_this_time: break
        if fp_resolved_this_time: resolved_fps_by_mcc_count += 1
    print(f"MCC Refinement FP Summary: Potentially Resolved FPs: {resolved_fps_by_mcc_count}/{len(target_fp_indices_for_mcc_sim)}")


# =====================================================================================
# --- SECTION C: MCC for False Negatives (Initial MCC checks for T1 & T3) ---
# =====================================================================================
print("\n\n--- SECTION C: MCC for False Negatives (Initial T1 & T3 MCC) ---")
if not (sim_enabled_mc_fwd or sim_enabled_mc_bwd):
    print("Adaptive epsilon for MCC (Fwd/Bwd) is disabled. Skipping Section C.")
else:
    target_fn_indices_for_mcc_sim = fn_indices_current_run
    print(f"Analyzing {len(target_fn_indices_for_mcc_sim)} FNs for initial MCC changes.")

    unblocked_fns_by_t1_mcc_count = 0
    unblocked_fns_by_t3_mcc_count = 0

    for pt_idx in target_fn_indices_for_mcc_sim:
        fn_data_w = traced_data_with_interp.get(pt_idx)
        if not fn_data_w: continue

        # di_base_mc_fwd_fn = base_eps_mc_fwd # Use configured base
        # di_base_mc_bwd_fn = base_eps_mc_bwd

        fn_unblocked_t1 = False
        fn_unblocked_t3 = False

        # Check T1 MCC
        t1_mcc_performed = fn_data_w.get("T1_mcc_performed", False)
        t1_mcc_result_bool = fn_data_w.get("T1_mcc_result_bool", None) # True if static
        if t1_mcc_performed and t1_mcc_result_bool == True: # If T1 MCC said it was static
            t1_mcc_debug = fn_data_w.get("T1_mcc_full_debug")
            if isinstance(t1_mcc_debug, dict):
                d_anchor_fn_t1 = t1_mcc_debug.get("d_anchor_of_point_global_for_mcc")
                # Iterate through di_details to find if any check would flip
                for di_detail in t1_mcc_debug.get("relevant_dis_details", []):
                    if not di_detail.get("match_found_in_di", False): continue # If it didn't match, adaptive won't make it static

                    # Check Interpolation Path
                    if di_detail.get("attempted_interpolation", False) and isinstance(di_detail.get("interp_depth_target_vs_surface"), dict):
                        interp_info = di_detail["interp_depth_target_vs_surface"]
                        target_d = interp_info.get("target_depth_in_hist_di")
                        surface_d = interp_info.get("interpolated_surface_depth")
                        if target_d is not None and surface_d is not None:
                            sim_eps_fwd = calculate_adaptive_epsilon_local(d_anchor_fn_t1, base_eps_mc_fwd, sim_dthr_mc_fwd, sim_kthr_mc_fwd, sim_dmax_mc_fwd, sim_dmin_mc_fwd, sim_enabled_mc_fwd)
                            sim_eps_bwd = calculate_adaptive_epsilon_local(d_anchor_fn_t1, base_eps_mc_bwd, sim_dthr_mc_bwd, sim_kthr_mc_bwd, sim_dmax_mc_bwd, sim_dmin_mc_bwd, sim_enabled_mc_bwd)
                            consistent_adaptive = (surface_d - sim_eps_bwd <= target_d <= surface_d + sim_eps_fwd)
                            if not consistent_adaptive: # If it's NO LONGER consistent
                                fn_unblocked_t1 = True
                                if diag_counters["mcc_fn_t1"] < MAX_DIAGNOSTICS_PER_SECTION:
                                    print_diag("mcc_fn_t1", f"FN Pt {pt_idx} (T1 MCC Interp): d_anchor={d_anchor_fn_t1:.2f}, target_d={target_d:.2f}, surface_d={surface_d:.2f}")
                                    print_diag("mcc_fn_t1", f"  SimEpsFwd={sim_eps_fwd:.3f}, SimEpsBwd={sim_eps_bwd:.3f}. OrigMatch=True, AdaptiveMatch={consistent_adaptive} -> UNBLOCKED")
                                    diag_counters["mcc_fn_t1"] += 1
                                break
                    if fn_unblocked_t1: break
                    # Check Direct Comparison Path
                    if di_detail.get("attempted_direct_comparison", False):
                        for static_cand_detail in di_detail.get("static_points_in_pixel_details", []):
                            if not static_cand_detail.get("match_found", False): continue
                            target_d = static_cand_detail.get("depth_target")
                            static_d = static_cand_detail.get("depth_static_cand")
                            if target_d is not None and static_d is not None and static_cand_detail.get("is_static_label"):
                                sim_eps_fwd = calculate_adaptive_epsilon_local(d_anchor_fn_t1, base_eps_mc_fwd, sim_dthr_mc_fwd, sim_kthr_mc_fwd, sim_dmax_mc_fwd, sim_dmin_mc_fwd, sim_enabled_mc_fwd)
                                sim_eps_bwd = calculate_adaptive_epsilon_local(d_anchor_fn_t1, base_eps_mc_bwd, sim_dthr_mc_bwd, sim_kthr_mc_bwd, sim_dmax_mc_bwd, sim_dmin_mc_bwd, sim_enabled_mc_bwd)
                                consistent_adaptive = (static_d - sim_eps_bwd <= target_d <= static_d + sim_eps_fwd)
                                if not consistent_adaptive:
                                    fn_unblocked_t1 = True
                                    if diag_counters["mcc_fn_t1"] < MAX_DIAGNOSTICS_PER_SECTION:
                                        print_diag("mcc_fn_t1", f"FN Pt {pt_idx} (T1 MCC Direct): d_anchor={d_anchor_fn_t1:.2f}, target_d={target_d:.2f}, static_d={static_d:.2f}")
                                        print_diag("mcc_fn_t1", f"  SimEpsFwd={sim_eps_fwd:.3f}, SimEpsBwd={sim_eps_bwd:.3f}. OrigMatch=True, AdaptiveMatch={consistent_adaptive} -> UNBLOCKED")
                                        diag_counters["mcc_fn_t1"] += 1
                                    break
                        if fn_unblocked_t1: break
        if fn_unblocked_t1: unblocked_fns_by_t1_mcc_count += 1


        # Check T3 Initial MCC
        t3_mcc_rejects_p = fn_data_w.get("T3_mcc_rejects_P", False) # True if P was rejected (deemed static)
        if t3_mcc_rejects_p:
            t3_mcc_debug = fn_data_w.get("T3_mcc_on_P_full_debug")
            if isinstance(t3_mcc_debug, dict):
                d_anchor_fn_t3 = t3_mcc_debug.get("d_anchor_of_point_global_for_mcc")
                # Iterate through di_details
                for di_detail in t3_mcc_debug.get("relevant_dis_details", []):
                    if not di_detail.get("match_found_in_di", False): continue

                    # Check Interpolation Path
                    if di_detail.get("attempted_interpolation", False) and isinstance(di_detail.get("interp_depth_target_vs_surface"), dict):
                        interp_info = di_detail["interp_depth_target_vs_surface"]
                        target_d = interp_info.get("target_depth_in_hist_di")
                        surface_d = interp_info.get("interpolated_surface_depth")
                        if target_d is not None and surface_d is not None:
                            sim_eps_fwd = calculate_adaptive_epsilon_local(d_anchor_fn_t3, base_eps_mc_fwd, sim_dthr_mc_fwd, sim_kthr_mc_fwd, sim_dmax_mc_fwd, sim_dmin_mc_fwd, sim_enabled_mc_fwd)
                            sim_eps_bwd = calculate_adaptive_epsilon_local(d_anchor_fn_t3, base_eps_mc_bwd, sim_dthr_mc_bwd, sim_kthr_mc_bwd, sim_dmax_mc_bwd, sim_dmin_mc_bwd, sim_enabled_mc_bwd)
                            consistent_adaptive = (surface_d - sim_eps_bwd <= target_d <= surface_d + sim_eps_fwd)
                            if not consistent_adaptive:
                                fn_unblocked_t3 = True
                                if diag_counters["mcc_fn_t3"] < MAX_DIAGNOSTICS_PER_SECTION:
                                    print_diag("mcc_fn_t3", f"FN Pt {pt_idx} (T3 Initial MCC Interp): d_anchor={d_anchor_fn_t3:.2f}, target_d={target_d:.2f}, surface_d={surface_d:.2f}")
                                    print_diag("mcc_fn_t3", f"  SimEpsFwd={sim_eps_fwd:.3f}, SimEpsBwd={sim_eps_bwd:.3f}. OrigMatch=True, AdaptiveMatch={consistent_adaptive} -> UNBLOCKED")
                                    diag_counters["mcc_fn_t3"] += 1
                                break
                    if fn_unblocked_t3: break
                    # Check Direct Comparison Path
                    if di_detail.get("attempted_direct_comparison", False):
                        for static_cand_detail in di_detail.get("static_points_in_pixel_details", []):
                            if not static_cand_detail.get("match_found", False): continue
                            target_d = static_cand_detail.get("depth_target")
                            static_d = static_cand_detail.get("depth_static_cand")
                            if target_d is not None and static_d is not None and static_cand_detail.get("is_static_label"):
                                sim_eps_fwd = calculate_adaptive_epsilon_local(d_anchor_fn_t3, base_eps_mc_fwd, sim_dthr_mc_fwd, sim_kthr_mc_fwd, sim_dmax_mc_fwd, sim_dmin_mc_fwd, sim_enabled_mc_fwd)
                                sim_eps_bwd = calculate_adaptive_epsilon_local(d_anchor_fn_t3, base_eps_mc_bwd, sim_dthr_mc_bwd, sim_kthr_mc_bwd, sim_dmax_mc_bwd, sim_dmin_mc_bwd, sim_enabled_mc_bwd)
                                consistent_adaptive = (static_d - sim_eps_bwd <= target_d <= static_d + sim_eps_fwd)
                                if not consistent_adaptive:
                                    fn_unblocked_t3 = True
                                    if diag_counters["mcc_fn_t3"] < MAX_DIAGNOSTICS_PER_SECTION:
                                        print_diag("mcc_fn_t3", f"FN Pt {pt_idx} (T3 Initial MCC Direct): d_anchor={d_anchor_fn_t3:.2f}, target_d={target_d:.2f}, static_d={static_d:.2f}")
                                        print_diag("mcc_fn_t3", f"  SimEpsFwd={sim_eps_fwd:.3f}, SimEpsBwd={sim_eps_bwd:.3f}. OrigMatch=True, AdaptiveMatch={consistent_adaptive} -> UNBLOCKED")
                                        diag_counters["mcc_fn_t3"] += 1
                                    break
                        if fn_unblocked_t3: break
        if fn_unblocked_t3: unblocked_fns_by_t3_mcc_count += 1
    
    print(f"Initial MCC FN Summary: FNs Unblocked by T1_MCC: {unblocked_fns_by_t1_mcc_count}/{len(target_fn_indices_for_mcc_sim)}")
    print(f"Initial MCC FN Summary: FNs Unblocked by T3_Initial_MCC: {unblocked_fns_by_t3_mcc_count}/{len(target_fn_indices_for_mcc_sim)}")

# %% Cell 3.12: Comprehensive Post-Hoc Analysis of Adaptive Epsilon Strategies (Extended Diagnostics for Sec B)

import pandas as pd
import numpy as np
from typing import Optional, Dict, Any # Ensure Optional is imported

print("\n--- Cell 3.12: Comprehensive Post-Hoc Analysis of Adaptive Epsilon Strategies ---")

# --- 0. Ensure Prerequisites ---
# (Same as your existing cell - config_accessor, debug_data_store, fp_indices_current_run, etc.)
if 'config_accessor' not in locals() or config_accessor is None:
    print("ERROR: config_accessor not found. Please run the configuration cell.")
    raise NameError("config_accessor not defined")
if 'debug_data_store' not in locals():
    print("ERROR: debug_data_store not found. Please run cells that populate it.")
    raise NameError("debug_data_store not defined")
if 'fp_indices_current_run' not in locals(): 
    print("WARNING: fp_indices_current_run not found. Analysis for FPs will be skipped.")
    fp_indices_current_run = []
if 'fn_indices_current_run' not in locals(): 
    print("WARNING: fn_indices_current_run not found. Analysis for FNs will be limited.")
    fn_indices_current_run = []
if 'df_test3_fails' not in locals(): 
     print("WARNING: df_test3_fails not found. Test 3 specific FN analysis might be incomplete.")
     df_test3_fails = pd.DataFrame() 

traced_data_with_interp = debug_data_store.get("with_interp", {})

# --- 1. Load ALL Adaptive Epsilon Simulation Parameters from Config ---
# (Same as your existing cell - loading for occ_depth, mc_fwd, mc_bwd)
occlusion_params_cfg = config_accessor.get_occlusion_determination_params()
adaptive_cfg_occ_depth = occlusion_params_cfg.get('adaptive_epsilon_depth_config', {})
base_eps_occ_depth = occlusion_params_cfg.get('epsilon_depth', 0.5) 

sim_enabled_occ = adaptive_cfg_occ_depth.get('enabled', False)
sim_dthr_occ = adaptive_cfg_occ_depth.get('dthr', 30.0)
sim_kthr_occ = adaptive_cfg_occ_depth.get('kthr', 0.05) 
sim_dmax_occ = adaptive_cfg_occ_depth.get('dmax', 2.0)
sim_dmin_occ = adaptive_cfg_occ_depth.get('dmin', 0.1)

mc_params_cfg = config_accessor.get_map_consistency_params()
adaptive_cfg_mc_fwd = mc_params_cfg.get('adaptive_epsilon_forward_config', {})
base_eps_mc_fwd = mc_params_cfg.get('epsilon_depth_forward_m', 1.0)

sim_enabled_mc_fwd = adaptive_cfg_mc_fwd.get('enabled', False)
sim_dthr_mc_fwd = adaptive_cfg_mc_fwd.get('dthr', 20.0)
sim_kthr_mc_fwd = adaptive_cfg_mc_fwd.get('kthr', 0.07)
sim_dmax_mc_fwd = adaptive_cfg_mc_fwd.get('dmax', 3.0)
sim_dmin_mc_fwd = adaptive_cfg_mc_fwd.get('dmin', 0.2)

adaptive_cfg_mc_bwd = mc_params_cfg.get('adaptive_epsilon_backward_config', {})
base_eps_mc_bwd = mc_params_cfg.get('epsilon_depth_backward_m', 0.3)

sim_enabled_mc_bwd = adaptive_cfg_mc_bwd.get('enabled', False)
sim_dthr_mc_bwd = adaptive_cfg_mc_bwd.get('dthr', 20.0)
sim_kthr_mc_bwd = adaptive_cfg_mc_bwd.get('kthr', 0.03)
sim_dmax_mc_bwd = adaptive_cfg_mc_bwd.get('dmax', 1.0)
sim_dmin_mc_bwd = adaptive_cfg_mc_bwd.get('dmin', 0.05)

print("Adaptive Epsilon Simulation Settings Loaded:")
print(f"  Occlusion Depth: enabled={sim_enabled_occ}, base_eps={base_eps_occ_depth:.2f}, dthr={sim_dthr_occ}, kthr={sim_kthr_occ}, dmax={sim_dmax_occ}, dmin={sim_dmin_occ}")
print(f"  MCC Forward:     enabled={sim_enabled_mc_fwd}, base_eps={base_eps_mc_fwd:.2f}, dthr={sim_dthr_mc_fwd}, kthr={sim_kthr_mc_fwd}, dmax={sim_dmax_mc_fwd}, dmin={sim_dmin_mc_fwd}")
print(f"  MCC Backward:    enabled={sim_enabled_mc_bwd}, base_eps={base_eps_mc_bwd:.2f}, dthr={sim_dthr_mc_bwd}, kthr={sim_kthr_mc_bwd}, dmax={sim_dmax_mc_bwd}, dmin={sim_dmin_mc_bwd}")

# --- 2. Define calculate_adaptive_epsilon locally ---
# (Same as your existing cell)
def calculate_adaptive_epsilon_local(
    d_anchor: Optional[float],
    di_base: float,
    dthr: float, kthr: float, dmax: float, dmin: float, 
    is_enabled: bool = True
) -> float:
    if not is_enabled or d_anchor is None or d_anchor < 0:
        return di_base
    adjustment = 0.0
    if kthr >= 0: 
        if d_anchor > dthr: adjustment = kthr * (d_anchor - dthr)
        calculated_epsilon = di_base + adjustment
    else: 
        if d_anchor > dthr: 
            adjustment = kthr * (d_anchor - dthr) 
            calculated_epsilon = di_base + adjustment
        else: 
            calculated_epsilon = di_base
    final_epsilon = np.clip(calculated_epsilon, dmin, dmax)
    return float(final_epsilon)

# --- Helper for printing diagnostics ---
MAX_DIAGNOSTICS_PER_SECTION = 3 # You can increase this to see more details
diag_counters = {"test3_fn": 0, "mcc_fp": 0, "mcc_fn_t1": 0, "mcc_fn_t3":0}

def print_diag(section_key, *args):
    # This function will now be called INSIDE the conditional print block
    # to ensure the counter is only incremented when a full diagnostic block is printed.
    print("    ", *args)

# =====================================================================================
# --- SECTION A: Test 3 "No Point Found Occluded" FNs ---
# =====================================================================================
# (Same as your existing cell - no changes needed here for this request)
print("\n\n--- SECTION A: Test 3 'No Point Found Occluded' FNs ---")
if not sim_enabled_occ:
    print("Adaptive epsilon for Occlusion Depth (Test 3) is disabled. Skipping Section A.")
else:
    target_fn_pt_indices_for_t3_sim = []
    if not df_test3_fails.empty:
        target_failure_reason_t3 = "Failed: No point found in historical DI pixel that was occluded by current chain point."
        fns_for_t3_sim_df = df_test3_fails[
            (df_test3_fails["T3_initial_MCC_rejected_P"] == False) &
            (df_test3_fails["T3_execution_reason_for_fail"] == target_failure_reason_t3)
        ]
        target_fn_pt_indices_for_t3_sim = fns_for_t3_sim_df["pt_idx"].tolist()
    print(f"Found {len(target_fn_pt_indices_for_t3_sim)} FNs where Test 3 failed due to '{target_failure_reason_t3}'.")

    if not target_fn_pt_indices_for_t3_sim:
        print("No FNs matching for Test 3 simulation.")
    else:
        resolved_t3_fns_count = 0; resolved_t3_links_count = 0
        for pt_idx in target_fn_pt_indices_for_t3_sim:
            fn_data_w = traced_data_with_interp.get(pt_idx); fn_resolved_this_time_t3 = False
            if not fn_data_w: continue
            test3_full_debug = fn_data_w.get("Test3_full_debug")
            if not isinstance(test3_full_debug, dict): continue
            di_base_t3 = base_eps_occ_depth
            steps_debug_list = test3_full_debug.get("steps_debug", [])
            for step_info in steps_debug_list:
                if step_info.get("status") == target_failure_reason_t3:
                    d_anchor_occluder_t3 = step_info.get("d_anchor_of_occluder_for_this_step")
                    occlusion_candidates = step_info.get("occlusion_check_details", [])
                    for cand_idx, cand_check in enumerate(occlusion_candidates):
                        if cand_check.get("forms_next_link_in_chain") == True: continue
                        depth_o = cand_check.get("depth_occluder_in_hist_frame"); depth_h = cand_check.get("depth_hist_cand_native_to_hist_frame")
                        if depth_o is not None and depth_h is not None:
                            delta_actual = depth_h - depth_o
                            fixed_eps_used_t3 = cand_check.get("epsilon_used_for_check", di_base_t3)
                            if diag_counters["test3_fn"] < MAX_DIAGNOSTICS_PER_SECTION and delta_actual <= 0:
                                print_diag("test3_fn", f"Pt {pt_idx}, T3 Step {step_info.get('step')}, Cand {cand_idx}: delta_actual={delta_actual:.3f} <= 0. Geom fail.")
                                diag_counters["test3_fn"] += 1
                            if delta_actual > 0:
                                sim_adaptive_eps_t3 = calculate_adaptive_epsilon_local(d_anchor_occluder_t3, di_base_t3, sim_dthr_occ, sim_kthr_occ, sim_dmax_occ, sim_dmin_occ, sim_enabled_occ)
                                would_pass_adaptive_t3 = depth_o < (depth_h - sim_adaptive_eps_t3)
                                if diag_counters["test3_fn"] < MAX_DIAGNOSTICS_PER_SECTION:
                                    print_diag("test3_fn", f"Pt {pt_idx}, T3 Step {step_info.get('step')}, Cand {cand_idx}:")
                                    print_diag("test3_fn", f"  d_anchor={d_anchor_occluder_t3 if d_anchor_occluder_t3 is not None else 'N/A'}m, di_base={di_base_t3:.2f}m, delta_actual={delta_actual:.3f}m")
                                    print_diag("test3_fn", f"  FixedEps: {fixed_eps_used_t3:.3f}. SimAdaptiveEps: {sim_adaptive_eps_t3:.3f}")
                                    print_diag("test3_fn", f"  Condition: {depth_o:.3f} < ({depth_h:.3f} - SimEps:{sim_adaptive_eps_t3:.3f}) = {depth_h - sim_adaptive_eps_t3:.3f} -> {would_pass_adaptive_t3}")
                                    diag_counters["test3_fn"] += 1
                                if would_pass_adaptive_t3: resolved_t3_links_count += 1; fn_resolved_this_time_t3 = True; break
                    if fn_resolved_this_time_t3: break
            if fn_resolved_this_time_t3: resolved_t3_fns_count += 1
        print(f"Test 3 FN Summary: Resolved Links: {resolved_t3_links_count}, Resolved FNs: {resolved_t3_fns_count}/{len(target_fn_pt_indices_for_t3_sim)}")

# =====================================================================================
# --- SECTION B: MCC for False Positives (Refinement) ---
# =====================================================================================
print("\n\n--- SECTION B: MCC for False Positives (Refinement) ---")
if not (sim_enabled_mc_fwd or sim_enabled_mc_bwd):
    print("Adaptive epsilon for MCC (Fwd/Bwd) is disabled. Skipping Section B.")
else:
    target_fp_indices_for_mcc_sim = fp_indices_current_run 
    print(f"Analyzing {len(target_fp_indices_for_mcc_sim)} FPs for MCC refinement changes.")
    resolved_fps_by_mcc_count = 0
    
    for pt_idx in target_fp_indices_for_mcc_sim:
        fp_data_w = traced_data_with_interp.get(pt_idx)
        if not fp_data_w: continue
        
        if not fp_data_w.get("MCC_Refinement_performed", False) or fp_data_w.get("MCC_Refinement_overrode_to_static", False):
            continue 

        mcc_refinement_debug = fp_data_w.get("MCC_Refinement_full_debug")
        if not isinstance(mcc_refinement_debug, dict): continue

        d_anchor_fp = mcc_refinement_debug.get("d_anchor_of_point_global_for_mcc")
        di_base_mc_fwd_fp = base_eps_mc_fwd 
        di_base_mc_bwd_fp = base_eps_mc_bwd

        fp_resolved_this_time = False
        # Iterate through each DI checked during MCC refinement for this FP
        for di_detail in mcc_refinement_debug.get("relevant_dis_details", []):
            if fp_resolved_this_time: break # Already resolved by a check in a previous DI for this FP

            # We are interested if *any* check that FAILED with fixed epsilon would PASS with adaptive
            # If match_found_in_di is True, it means the *overall* check for this DI passed.
            # We need to look at individual candidate checks if interpolation/direct comp failed.
            
            # --- Interpolation Path Diagnostics ---
            if di_detail.get("attempted_interpolation", False) and isinstance(di_detail.get("interp_depth_target_vs_surface"), dict):
                interp_info = di_detail["interp_depth_target_vs_surface"]
                original_interp_match = interp_info.get('was_consistent', False) # This should be False if MCC Refinement didn't catch the FP

                if not original_interp_match: # Only simulate if original interp check failed
                    target_d = interp_info.get("target_depth_in_hist_di")
                    surface_d = interp_info.get("interpolated_surface_depth")
                    
                    if target_d is not None and surface_d is not None:
                        sim_eps_fwd = calculate_adaptive_epsilon_local(d_anchor_fp, di_base_mc_fwd_fp, sim_dthr_mc_fwd, sim_kthr_mc_fwd, sim_dmax_mc_fwd, sim_dmin_mc_fwd, sim_enabled_mc_fwd)
                        sim_eps_bwd = calculate_adaptive_epsilon_local(d_anchor_fp, di_base_mc_bwd_fp, sim_dthr_mc_bwd, sim_kthr_mc_bwd, sim_dmax_mc_bwd, sim_dmin_mc_bwd, sim_enabled_mc_bwd)
                        
                        consistent_adaptive = (surface_d - sim_eps_bwd <= target_d <= surface_d + sim_eps_fwd)
                        
                        if diag_counters["mcc_fp"] < MAX_DIAGNOSTICS_PER_SECTION:
                            print_diag("mcc_fp", f"FP Pt {pt_idx} (Interp attempt on DI TS {di_detail.get('di_timestamp'):.0f}):")
                            print_diag("mcc_fp", f"  d_anchor={d_anchor_fp if d_anchor_fp is not None else 'N/A'}m, target_d={target_d:.2f}, surface_d={surface_d:.2f}")
                            print_diag("mcc_fp", f"  BaseEpsFwd={di_base_mc_fwd_fp:.3f}, SimEpsFwd={sim_eps_fwd:.3f}")
                            print_diag("mcc_fp", f"  BaseEpsBwd={di_base_mc_bwd_fp:.3f}, SimEpsBwd={sim_eps_bwd:.3f}")
                            print_diag("mcc_fp", f"  OrigInterpMatch={original_interp_match}, AdaptiveInterpMatch={consistent_adaptive}")
                            diag_counters["mcc_fp"] += 1

                        if consistent_adaptive:
                            fp_resolved_this_time = True; break 
            
            if fp_resolved_this_time: continue # Move to next FP if resolved by interpolation

            # --- Direct Comparison Path Diagnostics ---
            if di_detail.get("attempted_direct_comparison", False):
                for static_cand_detail in di_detail.get("static_points_in_pixel_details", []):
                    original_direct_match_cand = static_cand_detail.get("match_found", False) # Check for this specific candidate

                    if not original_direct_match_cand: # Only simulate if this candidate didn't match
                        target_d = static_cand_detail.get("depth_target")
                        static_d = static_cand_detail.get("depth_static_cand")

                        if target_d is not None and static_d is not None and static_cand_detail.get("is_static_label"):
                            sim_eps_fwd = calculate_adaptive_epsilon_local(d_anchor_fp, di_base_mc_fwd_fp, sim_dthr_mc_fwd, sim_kthr_mc_fwd, sim_dmax_mc_fwd, sim_dmin_mc_fwd, sim_enabled_mc_fwd)
                            sim_eps_bwd = calculate_adaptive_epsilon_local(d_anchor_fp, di_base_mc_bwd_fp, sim_dthr_mc_bwd, sim_kthr_mc_bwd, sim_dmax_mc_bwd, sim_dmin_mc_bwd, sim_enabled_mc_bwd)

                            consistent_adaptive = (static_d - sim_eps_bwd <= target_d <= static_d + sim_eps_fwd)
                            
                            if diag_counters["mcc_fp"] < MAX_DIAGNOSTICS_PER_SECTION:
                                print_diag("mcc_fp", f"FP Pt {pt_idx} (DirectCand TS {di_detail.get('di_timestamp'):.0f}, OrigIdx {static_cand_detail.get('original_index_in_di')}):")
                                print_diag("mcc_fp", f"  d_anchor={d_anchor_fp if d_anchor_fp is not None else 'N/A'}m, target_d={target_d:.2f}, static_d={static_d:.2f}")
                                print_diag("mcc_fp", f"  BaseEpsFwd={di_base_mc_fwd_fp:.3f}, SimEpsFwd={sim_eps_fwd:.3f}")
                                print_diag("mcc_fp", f"  BaseEpsBwd={di_base_mc_bwd_fp:.3f}, SimEpsBwd={sim_eps_bwd:.3f}")
                                print_diag("mcc_fp", f"  OrigDirectMatchCand={original_direct_match_cand}, AdaptiveDirectMatchCand={consistent_adaptive}")
                                diag_counters["mcc_fp"] += 1
                                
                            if consistent_adaptive:
                                fp_resolved_this_time = True; break
                if fp_resolved_this_time: break 
        
        if fp_resolved_this_time: resolved_fps_by_mcc_count += 1
    print(f"MCC Refinement FP Summary: Potentially Resolved FPs: {resolved_fps_by_mcc_count}/{len(target_fp_indices_for_mcc_sim)}")

# =====================================================================================
# --- SECTION C: MCC for False Negatives (Initial T1 & T3 MCC) ---
# =====================================================================================
# (Same as your existing cell - no changes needed here for this request, 
#  assuming its diagnostics were sufficient from the previous output)
print("\n\n--- SECTION C: MCC for False Negatives (Initial T1 & T3 MCC) ---")
if not (sim_enabled_mc_fwd or sim_enabled_mc_bwd):
    print("Adaptive epsilon for MCC (Fwd/Bwd) is disabled. Skipping Section C.")
else:
    target_fn_indices_for_mcc_sim = fn_indices_current_run
    print(f"Analyzing {len(target_fn_indices_for_mcc_sim)} FNs for initial MCC changes.")
    unblocked_fns_by_t1_mcc_count = 0; unblocked_fns_by_t3_mcc_count = 0
    for pt_idx in target_fn_indices_for_mcc_sim:
        fn_data_w = traced_data_with_interp.get(pt_idx)
        if not fn_data_w: continue
        fn_unblocked_t1 = False; fn_unblocked_t3 = False
        t1_mcc_performed = fn_data_w.get("T1_mcc_performed", False); t1_mcc_result_bool = fn_data_w.get("T1_mcc_result_bool", None)
        if t1_mcc_performed and t1_mcc_result_bool == True:
            t1_mcc_debug = fn_data_w.get("T1_mcc_full_debug")
            if isinstance(t1_mcc_debug, dict):
                d_anchor_fn_t1 = t1_mcc_debug.get("d_anchor_of_point_global_for_mcc")
                for di_detail in t1_mcc_debug.get("relevant_dis_details", []):
                    if not di_detail.get("match_found_in_di", False): continue
                    if di_detail.get("attempted_interpolation", False) and isinstance(di_detail.get("interp_depth_target_vs_surface"), dict):
                        interp_info = di_detail["interp_depth_target_vs_surface"]; target_d = interp_info.get("target_depth_in_hist_di"); surface_d = interp_info.get("interpolated_surface_depth")
                        if target_d is not None and surface_d is not None:
                            sim_eps_fwd = calculate_adaptive_epsilon_local(d_anchor_fn_t1, base_eps_mc_fwd, sim_dthr_mc_fwd, sim_kthr_mc_fwd, sim_dmax_mc_fwd, sim_dmin_mc_fwd, sim_enabled_mc_fwd)
                            sim_eps_bwd = calculate_adaptive_epsilon_local(d_anchor_fn_t1, base_eps_mc_bwd, sim_dthr_mc_bwd, sim_kthr_mc_bwd, sim_dmax_mc_bwd, sim_dmin_mc_bwd, sim_enabled_mc_bwd)
                            consistent_adaptive = (surface_d - sim_eps_bwd <= target_d <= surface_d + sim_eps_fwd)
                            if not consistent_adaptive:
                                fn_unblocked_t1 = True
                                if diag_counters["mcc_fn_t1"] < MAX_DIAGNOSTICS_PER_SECTION:
                                    print_diag("mcc_fn_t1", f"FN Pt {pt_idx} (T1 MCC Interp Unblock): d_anchor={d_anchor_fn_t1 if d_anchor_fn_t1 is not None else 'N/A'}m, target_d={target_d:.2f}, surface_d={surface_d:.2f}")
                                    print_diag("mcc_fn_t1", f"  SimEpsFwd={sim_eps_fwd:.3f}, SimEpsBwd={sim_eps_bwd:.3f}. OrigMatch=True, AdaptiveMatch={consistent_adaptive} -> UNBLOCKED")
                                    diag_counters["mcc_fn_t1"] += 1
                                break
                    if fn_unblocked_t1: break
                    if di_detail.get("attempted_direct_comparison", False):
                        for static_cand_detail in di_detail.get("static_points_in_pixel_details", []):
                            if not static_cand_detail.get("match_found", False): continue
                            target_d = static_cand_detail.get("depth_target"); static_d = static_cand_detail.get("depth_static_cand")
                            if target_d is not None and static_d is not None and static_cand_detail.get("is_static_label"):
                                sim_eps_fwd = calculate_adaptive_epsilon_local(d_anchor_fn_t1, base_eps_mc_fwd, sim_dthr_mc_fwd, sim_kthr_mc_fwd, sim_dmax_mc_fwd, sim_dmin_mc_fwd, sim_enabled_mc_fwd)
                                sim_eps_bwd = calculate_adaptive_epsilon_local(d_anchor_fn_t1, base_eps_mc_bwd, sim_dthr_mc_bwd, sim_kthr_mc_bwd, sim_dmax_mc_bwd, sim_dmin_mc_bwd, sim_enabled_mc_bwd)
                                consistent_adaptive = (static_d - sim_eps_bwd <= target_d <= static_d + sim_eps_fwd)
                                if not consistent_adaptive:
                                    fn_unblocked_t1 = True
                                    if diag_counters["mcc_fn_t1"] < MAX_DIAGNOSTICS_PER_SECTION:
                                        print_diag("mcc_fn_t1", f"FN Pt {pt_idx} (T1 MCC Direct Unblock): d_anchor={d_anchor_fn_t1 if d_anchor_fn_t1 is not None else 'N/A'}m, target_d={target_d:.2f}, static_d={static_d:.2f}")
                                        print_diag("mcc_fn_t1", f"  SimEpsFwd={sim_eps_fwd:.3f}, SimEpsBwd={sim_eps_bwd:.3f}. OrigMatch=True, AdaptiveMatch={consistent_adaptive} -> UNBLOCKED")
                                        diag_counters["mcc_fn_t1"] += 1
                                    break
                        if fn_unblocked_t1: break
        if fn_unblocked_t1: unblocked_fns_by_t1_mcc_count += 1
        t3_mcc_rejects_p = fn_data_w.get("T3_mcc_rejects_P", False)
        if t3_mcc_rejects_p:
            t3_mcc_debug = fn_data_w.get("T3_mcc_on_P_full_debug")
            if isinstance(t3_mcc_debug, dict):
                d_anchor_fn_t3 = t3_mcc_debug.get("d_anchor_of_point_global_for_mcc")
                for di_detail in t3_mcc_debug.get("relevant_dis_details", []):
                    if not di_detail.get("match_found_in_di", False): continue
                    if di_detail.get("attempted_interpolation", False) and isinstance(di_detail.get("interp_depth_target_vs_surface"), dict):
                        interp_info = di_detail["interp_depth_target_vs_surface"]; target_d = interp_info.get("target_depth_in_hist_di"); surface_d = interp_info.get("interpolated_surface_depth")
                        if target_d is not None and surface_d is not None:
                            sim_eps_fwd = calculate_adaptive_epsilon_local(d_anchor_fn_t3, base_eps_mc_fwd, sim_dthr_mc_fwd, sim_kthr_mc_fwd, sim_dmax_mc_fwd, sim_dmin_mc_fwd, sim_enabled_mc_fwd)
                            sim_eps_bwd = calculate_adaptive_epsilon_local(d_anchor_fn_t3, base_eps_mc_bwd, sim_dthr_mc_bwd, sim_kthr_mc_bwd, sim_dmax_mc_bwd, sim_dmin_mc_bwd, sim_enabled_mc_bwd)
                            consistent_adaptive = (surface_d - sim_eps_bwd <= target_d <= surface_d + sim_eps_fwd)
                            if not consistent_adaptive:
                                fn_unblocked_t3 = True
                                if diag_counters["mcc_fn_t3"] < MAX_DIAGNOSTICS_PER_SECTION:
                                    print_diag("mcc_fn_t3", f"FN Pt {pt_idx} (T3 Initial MCC Interp Unblock): d_anchor={d_anchor_fn_t3 if d_anchor_fn_t3 is not None else 'N/A'}m, target_d={target_d:.2f}, surface_d={surface_d:.2f}")
                                    print_diag("mcc_fn_t3", f"  SimEpsFwd={sim_eps_fwd:.3f}, SimEpsBwd={sim_eps_bwd:.3f}. OrigMatch=True, AdaptiveMatch={consistent_adaptive} -> UNBLOCKED")
                                    diag_counters["mcc_fn_t3"] += 1
                                break
                    if fn_unblocked_t3: break
                    if di_detail.get("attempted_direct_comparison", False):
                        for static_cand_detail in di_detail.get("static_points_in_pixel_details", []):
                            if not static_cand_detail.get("match_found", False): continue
                            target_d = static_cand_detail.get("depth_target"); static_d = static_cand_detail.get("depth_static_cand")
                            if target_d is not None and static_d is not None and static_cand_detail.get("is_static_label"):
                                sim_eps_fwd = calculate_adaptive_epsilon_local(d_anchor_fn_t3, base_eps_mc_fwd, sim_dthr_mc_fwd, sim_kthr_mc_fwd, sim_dmax_mc_fwd, sim_dmin_mc_fwd, sim_enabled_mc_fwd)
                                sim_eps_bwd = calculate_adaptive_epsilon_local(d_anchor_fn_t3, base_eps_mc_bwd, sim_dthr_mc_bwd, sim_kthr_mc_bwd, sim_dmax_mc_bwd, sim_dmin_mc_bwd, sim_enabled_mc_bwd)
                                consistent_adaptive = (static_d - sim_eps_bwd <= target_d <= static_d + sim_eps_fwd)
                                if not consistent_adaptive:
                                    fn_unblocked_t3 = True
                                    if diag_counters["mcc_fn_t3"] < MAX_DIAGNOSTICS_PER_SECTION:
                                        print_diag("mcc_fn_t3", f"FN Pt {pt_idx} (T3 Initial MCC Direct Unblock): d_anchor={d_anchor_fn_t3 if d_anchor_fn_t3 is not None else 'N/A'}m, target_d={target_d:.2f}, static_d={static_d:.2f}")
                                        print_diag("mcc_fn_t3", f"  SimEpsFwd={sim_eps_fwd:.3f}, SimEpsBwd={sim_eps_bwd:.3f}. OrigMatch=True, AdaptiveMatch={consistent_adaptive} -> UNBLOCKED")
                                        diag_counters["mcc_fn_t3"] += 1
                                    break
                        if fn_unblocked_t3: break
        if fn_unblocked_t3: unblocked_fns_by_t3_mcc_count += 1
    print(f"Initial MCC FN Summary: FNs Unblocked by T1_MCC: {unblocked_fns_by_t1_mcc_count}/{len(target_fn_indices_for_mcc_sim)}")
    print(f"Initial MCC FN Summary: FNs Unblocked by T3_Initial_MCC: {unblocked_fns_by_t3_mcc_count}/{len(target_fn_indices_for_mcc_sim)}")

# =====================================================================================
# --- SECTION D: Test 2 "Parallel Motion" FNs & FPs (detailed_check_epsilon_depth) ---
# =====================================================================================
print("\n\n--- SECTION D: Test 2 'Parallel Motion' FNs & FPs ---")
if not sim_enabled_occ: # Test 2 uses the same adaptive config as Test 3 for now
    print("Adaptive epsilon for Occlusion Depth (used by Test 2) is disabled. Skipping Section D.")
else:
    # --- D.1: Analyze FNs where Test 2 might have failed ---
    target_fn_indices_for_t2_sim = fn_indices_current_run # Or a subset if you have specific Test 2 failure flags
    print(f"Analyzing {len(target_fn_indices_for_t2_sim)} FNs for Test 2 changes.")
    
    resolved_fns_by_t2_count = 0
    diag_counters["test2_fn"] = 0 # Reset/initialize counter for this sub-section

    for pt_idx in target_fn_indices_for_t2_sim:
        fn_data_w = traced_data_with_interp.get(pt_idx)
        if not fn_data_w: continue
        
        test2_full_debug = fn_data_w.get("Test2_full_debug")
        if not isinstance(test2_full_debug, dict): continue

        original_test2_passed = test2_full_debug.get("test_passed", False)
        if original_test2_passed: # If Test 2 already passed, it didn't cause this FN
            continue

        # Test 2 failed for this FN. Let's see if adaptive epsilon could make it pass.
        # This requires re-simulating the chain. For simplicity, we'll check the first failing step.
        
        simulated_chain_passes_t2 = True # Assume it passes until a step fails with adaptive
        current_sim_d_anchor = test2_full_debug.get("initial_d_anchor_of_P") # d_anchor of P

        for step_idx, step_info in enumerate(test2_full_debug.get("steps_debug", [])):
            original_step_status = step_info.get("status", "")
            sim_step_passed_this_time = False
            
            # If original step already succeeded, assume it would with adaptive too for this simplified check
            if "Success" in original_step_status:
                # Update d_anchor for the next simulated step based on the original successful link
                if step_info.get("d_anchor_for_next_pto_step") is not None:
                    current_sim_d_anchor = step_info.get("d_anchor_for_next_pto_step")
                sim_step_passed_this_time = True
                # continue # to next step in chain
            
            # If original step failed, check if adaptive epsilon helps any candidate
            elif "Failed" in original_step_status :
                for cand_idx, cand_check in enumerate(step_info.get("occlusion_check_details", [])):
                    detailed_results = cand_check.get("detailed_occlusion_check_results", {})
                    if not detailed_results or detailed_results.get("error"): continue

                    d_eval_hist = detailed_results.get("d_eval_in_hist_frame")
                    d_hist_cand_hist = detailed_results.get("d_hist_cand_in_hist_frame")
                    # d_anchor_for_current_step_pto = step_info.get("d_anchor_of_pto") # This is what was used
                    d_anchor_for_current_step_pto = current_sim_d_anchor # More accurate for chain simulation

                    if d_eval_hist is None or d_hist_cand_hist is None: continue

                    # Base epsilon for detailed check (from config, should match what's in detailed_results)
                    base_eps_t2_detailed = config_accessor.get_occlusion_determination_params().get('epsilon_depth', 0.5) 
                                            # Or: mdetector_instance.detailed_check_epsilon_depth if available

                    sim_adaptive_eps_t2 = calculate_adaptive_epsilon_local(
                        d_anchor_for_current_step_pto, base_eps_t2_detailed,
                        sim_dthr_occ, sim_kthr_occ, sim_dmax_occ, sim_dmin_occ,
                        sim_enabled_occ
                    )
                    
                    # Test 2 condition: point_eval is OCCLUDED_BY point_hist_cand
                    # d_eval_in_hist > d_hist_cand_in_hist + epsilon
                    would_pass_adaptive_link = d_eval_hist > (d_hist_cand_hist + sim_adaptive_eps_t2)
                    
                    original_link_passed = detailed_results.get("depth_condition_met", False) and \
                                           detailed_results.get("occlusion_type_checked") == "OCCLUDED_BY" and \
                                           not cand_check.get("mcc_rejects_this_link", False)


                    if diag_counters["test2_fn"] < MAX_DIAGNOSTICS_PER_SECTION and not original_link_passed:
                        print_diag("test2_fn", f"FN Pt {pt_idx}, T2 Step {step_idx}, Cand {cand_idx} (Orig PTO d_anchor {d_anchor_for_current_step_pto:.2f}):")
                        print_diag("test2_fn", f"  d_eval_h={d_eval_hist:.2f}, d_hist_h={d_hist_cand_hist:.2f}, BaseEps={base_eps_t2_detailed:.2f}, SimAdapEps={sim_adaptive_eps_t2:.2f}")
                        print_diag("test2_fn", f"  OrigLinkPassed={original_link_passed}, AdaptiveLinkWouldPass={would_pass_adaptive_link}")
                        # Increment only if we actually print a candidate that could flip
                        if not original_link_passed and would_pass_adaptive_link : diag_counters["test2_fn"] +=1


                    if would_pass_adaptive_link and not cand_check.get("mcc_rejects_this_link", False):
                        sim_step_passed_this_time = True
                        # Update d_anchor for the next simulated step
                        if historical_di_k_local_sph_coords := cand_check.get("detailed_occlusion_check_results",{}).get("sph_coords_of_hist_cand_in_hist_frame"): # This is not directly available, need to fetch from DI
                             # This is tricky: need original_idx_hist_cand and historical_di_k to get its native depth
                             # For a simpler simulation, if a link passes, assume the d_anchor for next step is based on this point_hist_cand_global
                             # We'd need to project point_hist_cand_global into its *own* frame (historical_di_k) to get its native depth.
                             # This is complex for post-hoc. Let's assume if one link passes, the step passes.
                             # And for d_anchor propagation, use the one from the original trace if the step was originally successful.
                             # This part of d_anchor propagation in pure simulation is hard without re-running MDet logic.
                             # For now, if a failing step is fixed, we count the FN.
                            pass
                        break # This candidate makes the step pass

            if not sim_step_passed_this_time:
                simulated_chain_passes_t2 = False # This step failed with adaptive, so chain fails
                break # No need to check further steps in this chain

        if simulated_chain_passes_t2 and not original_test2_passed : # If original failed but sim passes
            resolved_fns_by_t2_count += 1
            if diag_counters["test2_fn"] < MAX_DIAGNOSTICS_PER_SECTION : # Print if FN resolved and haven't printed max
                 print_diag("test2_fn", f"  --> FN Pt {pt_idx} Test 2 chain would PASS with adaptive epsilon.")
                 # diag_counters["test2_fn"] +=1 # Already incremented for candidate
    
    print(f"Test 2 FN Summary: Potentially Resolved FNs where Test 2 originally failed: {resolved_fns_by_t2_count}/{len(target_fn_indices_for_t2_sim)}")

    # --- D.2: Analyze FPs where Test 2 might have passed incorrectly ---
    target_fp_indices_for_t2_sim = fp_indices_current_run
    print(f"Analyzing {len(target_fp_indices_for_t2_sim)} FPs for Test 2 changes.")
    corrected_fps_by_t2_count = 0
    diag_counters["test2_fp"] = 0

    for pt_idx in target_fp_indices_for_t2_sim:
        fp_data_w = traced_data_with_interp.get(pt_idx)
        if not fp_data_w: continue

        test2_full_debug = fp_data_w.get("Test2_full_debug")
        if not isinstance(test2_full_debug, dict): continue

        original_test2_passed = test2_full_debug.get("test_passed", False)
        if not original_test2_passed: # If Test 2 already failed, it didn't cause this FP
            continue
        
        # Test 2 passed for this FP. Let's see if adaptive epsilon could make it fail.
        simulated_chain_still_passes_t2 = True # Assume it still passes
        current_sim_d_anchor_fp = test2_full_debug.get("initial_d_anchor_of_P")

        for step_idx, step_info in enumerate(test2_full_debug.get("steps_debug", [])):
            # We are looking for a step that originally PASSED but would FAIL with adaptive epsilon
            sim_step_fails_this_time = False
            
            for cand_idx, cand_check in enumerate(step_info.get("occlusion_check_details", [])):
                if not cand_check.get("forms_next_link_in_chain", False): continue # This candidate wasn't part of the successful chain

                detailed_results = cand_check.get("detailed_occlusion_check_results", {})
                if not detailed_results or detailed_results.get("error"): continue

                d_eval_hist = detailed_results.get("d_eval_in_hist_frame")
                d_hist_cand_hist = detailed_results.get("d_hist_cand_in_hist_frame")
                # d_anchor_for_current_step_pto = step_info.get("d_anchor_of_pto")
                d_anchor_for_current_step_pto = current_sim_d_anchor_fp


                if d_eval_hist is None or d_hist_cand_hist is None: continue
                
                base_eps_t2_detailed = config_accessor.get_occlusion_determination_params().get('epsilon_depth', 0.5)

                sim_adaptive_eps_t2 = calculate_adaptive_epsilon_local(
                    d_anchor_for_current_step_pto, base_eps_t2_detailed,
                    sim_dthr_occ, sim_kthr_occ, sim_dmax_occ, sim_dmin_occ,
                    sim_enabled_occ
                )
                
                would_pass_adaptive_link = d_eval_hist > (d_hist_cand_hist + sim_adaptive_eps_t2)
                original_link_passed = detailed_results.get("depth_condition_met", False) and \
                                       detailed_results.get("occlusion_type_checked") == "OCCLUDED_BY" and \
                                       not cand_check.get("mcc_rejects_this_link", False)

                if diag_counters["test2_fp"] < MAX_DIAGNOSTICS_PER_SECTION and original_link_passed:
                    print_diag("test2_fp", f"FP Pt {pt_idx}, T2 Step {step_idx}, Cand {cand_idx} (Orig PTO d_anchor {d_anchor_for_current_step_pto:.2f}):")
                    print_diag("test2_fp", f"  d_eval_h={d_eval_hist:.2f}, d_hist_h={d_hist_cand_hist:.2f}, BaseEps={base_eps_t2_detailed:.2f}, SimAdapEps={sim_adaptive_eps_t2:.2f}")
                    print_diag("test2_fp", f"  OrigLinkPassed={original_link_passed}, AdaptiveLinkWouldPass={would_pass_adaptive_link}")
                    if original_link_passed and not would_pass_adaptive_link: diag_counters["test2_fp"] +=1


                if original_link_passed and not would_pass_adaptive_link:
                    sim_step_fails_this_time = True # This critical link failed with adaptive
                    break 
            
            if sim_step_fails_this_time:
                simulated_chain_still_passes_t2 = False # This step failed, so chain fails
                break
            else: # Step still passes or no critical link failed
                # Update d_anchor for next simulated step if this step was part of original success
                if step_info.get("d_anchor_for_next_pto_step") is not None:
                     current_sim_d_anchor_fp = step_info.get("d_anchor_for_next_pto_step")


        if not simulated_chain_still_passes_t2 and original_test2_passed: # Original passed, but sim fails
            corrected_fps_by_t2_count += 1
            if diag_counters["test2_fp"] < MAX_DIAGNOSTICS_PER_SECTION :
                 print_diag("test2_fp", f"  --> FP Pt {pt_idx} Test 2 chain would FAIL with adaptive epsilon (potential FP correction).")
                 # diag_counters["test2_fp"] +=1 # Already incremented

    print(f"Test 2 FP Summary: Potentially Corrected FPs where Test 2 originally passed: {corrected_fps_by_t2_count}/{len(target_fp_indices_for_t2_sim)}")


Pt 8086: T3_MCC_on_P outcome changed. With: True, Without: False
  Reason With: Consistent static points found in 2/9 DIs where checks were performed. Mode: Count, Threshold: 2. Met consistency thresholds.
    Interp Depth With: 20.870155041806147
  Reason Without: Consistent static points found in 0/9 DIs where checks were performed. Mode: Count, Threshold: 2. Did NOT meet consistency thresholds.
  >>> But FINAL LABEL DID NOT CHANGE. With: UNDETERMINED, Without: UNDETERMINED
      Intermediate outcomes WITH: T1/4=UNDETERMINED, T2=UNDETERMINED, T3=UNDETERMINED
      Intermediate outcomes WITHOUT: T1/4=UNDETERMINED, T2=UNDETERMINED, T3=UNDETERMINED

Pt 8687: T3_MCC_on_P outcome changed. With: True, Without: False
  Reason With: Consistent static points found in 2/9 DIs where checks were performed. Mode: Count, Threshold: 2. Met consistency thresholds.
    Interp Depth With: None
  Reason Without: Consistent static points found in 1/9 DIs where checks were performed. Mode: Count, Thresh

# Chapter 4

In [ ]:
# %% Cell 4: Select Point and Historical DI for Interpolation Debug



# Increase logging here:
logging.getLogger('src.core.m_detector.map_consistency').setLevel(logging.DEBUG) # MCC logs
logging.getLogger('src.core.m_detector.interpolation_utils').setLevel(logging.DEBUG) # Interpolation logs

# %% Cell 4: Load GT for Target, Identify Misclassifications, and Select Point for Interpolation Debug

# --- User Input for this Cell ---
# Define the distance range from the sensor for selecting a debug point
# This is the range of P_current's depth *relative to its own sensor*
MIN_DIST_FOR_FP_SELECTION = 15.0  # meters
MAX_DIST_FOR_FP_SELECTION = 50.0  # meters

# Define the target depth range *for triggering interpolation* when P_current is projected into DI_hist
# These should align with or be informed by your config's interpolation_min/max_depth_m
# Forcing interpolation for this debug point:
FORCE_INTERPOLATION_FOR_SELECTED_POINT = True # Set to True to try and find a point that WILL attempt interpolation
TARGET_INTERP_MIN_DEPTH_IN_HIST_DI = config_accessor.get_map_consistency_params().get('interpolation_min_depth_m', 25.0)
TARGET_INTERP_MAX_DEPTH_IN_HIST_DI = config_accessor.get_map_consistency_params().get('interpolation_max_depth_m', 70.0)


RELATIVE_HIST_DI_IDX_FOR_CHECK = -1 # Check against immediate past
# --- End User Input ---

# ... (initializations for gt_indices_dict_for_target_interp, etc. as before) ...
gt_indices_dict_for_target_interp: Optional[Dict[str, Any]] = None
misclassified_summary_interp: Optional[Dict[str, List[int]]] = None
PT_IDX_FOR_INTERP_DEBUG: Optional[int] = None
point_global_for_interp_debug: Optional[np.ndarray] = None
historical_di_for_interp_check: Optional[DepthImage] = None
sph_coords_of_pt_in_hist_di: Optional[np.ndarray] = None
pixel_indices_of_pt_in_hist_di: Optional[Tuple[int,int]] = None


if nusc and target_scene_rec_interp and TARGET_LIDAR_SD_TOKEN_INTERP and \
   processed_target_di_interp and processed_target_di_interp.original_points_global_coords is not None and \
   config_accessor and detector_interp and idx_of_processed_target_di_interp is not None:

    # 1. Load GT Labels (same as your current Cell 4)
    # ... (your GT loading logic) ...
    print(f"\nLoading Ground Truth labels for the target sweep (Token: {TARGET_LIDAR_SD_TOKEN_INTERP[:8]}) for analysis...")
    gt_labels_hdf5_dir_interp = config_accessor.get_nuscenes_params().get('label_path')
    if gt_labels_hdf5_dir_interp and not os.path.isabs(gt_labels_hdf5_dir_interp):
        gt_labels_hdf5_dir_interp = os.path.join(PROJECT_ROOT, gt_labels_hdf5_dir_interp)
    
    gt_labels_scene_hdf5_filepath_interp = os.path.join(gt_labels_hdf5_dir_interp, f"gt_point_labels_{target_scene_rec_interp['name']}.h5")
    
    validation_params_interp = config_accessor.get_validation_params()
    velocity_threshold_gt_interp = validation_params_interp.get('gt_velocity_threshold', 0.5)
    
    point_pre_filtering_params_interp = detector_interp.config_accessor.get_point_pre_filtering_params()
    mdet_min_range_interp = point_pre_filtering_params_interp.get('min_range_meters', 1.0)
    mdet_max_range_interp = point_pre_filtering_params_interp.get('max_range_meters', 80.0)

    if os.path.exists(gt_labels_scene_hdf5_filepath_interp):
        gt_indices_dict_for_target_interp = get_gt_dynamic_points_for_sweep(
            nusc=nusc,
            sweep_data_dict=target_sweep_data_dict_interp,
            all_points_global_mdetector=processed_target_di_interp.original_points_global_coords,
            gt_labels_scene_hdf5_path=gt_labels_scene_hdf5_filepath_interp,
            velocity_threshold=velocity_threshold_gt_interp,
            mdetector_min_range=mdet_min_range_interp,
            mdetector_max_range=mdet_max_range_interp
        )
        if not gt_indices_dict_for_target_interp or gt_indices_dict_for_target_interp.get('error_msg'):
            print(f"Error loading GT for target sweep: {gt_indices_dict_for_target_interp.get('error_msg', 'Unknown GT loading error')}")
            gt_indices_dict_for_target_interp = None
    else:
        print(f"ERROR: GT HDF5 file not found for interpolation debug: {gt_labels_scene_hdf5_filepath_interp}")

    # 2. Identify Misclassifications (same as your current Cell 4)
    if gt_indices_dict_for_target_interp and processed_target_di_interp.mdet_labels_for_points is not None:
        print("\nIdentifying misclassifications for the target sweep...")
        misclassified_summary_interp = get_misclassified_points(
            processed_target_di_interp, 
            gt_indices_dict_for_target_interp
        )
        if misclassified_summary_interp:
            print(f"  Found {len(misclassified_summary_interp.get('fp_dynamic',[]))} FPs (Static/Unlabeled -> MDet Dynamic)")
            print(f"  Found {len(misclassified_summary_interp.get('fn_dynamic',[]))} FNs (Dynamic -> MDet Not Dynamic)")
    else:
        print("Could not identify misclassifications (GT not loaded or MDet labels missing).")

    # 3. Select a Point for Debugging, considering interpolation trigger depth
    candidate_debug_indices = []
    
    # First, determine the historical DI we will check against
    actual_hist_di_lib_idx_for_select = idx_of_processed_target_di_interp + RELATIVE_HIST_DI_IDX_FOR_CHECK
    temp_historical_di_for_select: Optional[DepthImage] = None
    if 0 <= actual_hist_di_lib_idx_for_select < len(detector_interp.depth_image_library.get_all_images()):
        temp_historical_di_for_select = detector_interp.depth_image_library.get_image_by_index(actual_hist_di_lib_idx_for_select)

    if misclassified_summary_interp and temp_historical_di_for_select:
        fp_indices = misclassified_summary_interp.get('fp_dynamic', [])
        if fp_indices:
            print(f"\nConsidering {len(fp_indices)} False Positive points for analysis.")
            
            # Get global coords and project FPs into the selected historical DI to check their depth *there*
            fp_points_global = processed_target_di_interp.original_points_global_coords[fp_indices, :3]
            
            _, fp_sph_coords_in_hist, _, fp_valid_proj_mask_in_hist = \
                temp_historical_di_for_select.project_points_batch(fp_points_global)
            
            fp_depths_in_hist_di = fp_sph_coords_in_hist[fp_valid_proj_mask_in_hist, 2]
            original_fp_indices_projected = np.array(fp_indices)[fp_valid_proj_mask_in_hist]

            # Filter by M-Detector's own range (distance from its own sensor)
            sensor_origin_global_current = target_sweep_data_dict_interp['T_global_lidar'][:3, 3]
            distances_to_current_sensor = np.linalg.norm(fp_points_global[fp_valid_proj_mask_in_hist] - sensor_origin_global_current.reshape(1,3), axis=1)
            
            current_sensor_dist_mask = (distances_to_current_sensor >= MIN_DIST_FOR_FP_SELECTION) & \
                                       (distances_to_current_sensor <= MAX_DIST_FOR_FP_SELECTION)

            # Filter by depth in historical DI (for interpolation triggering)
            if FORCE_INTERPOLATION_FOR_SELECTED_POINT:
                hist_di_depth_mask = (fp_depths_in_hist_di >= TARGET_INTERP_MIN_DEPTH_IN_HIST_DI) & \
                                     (fp_depths_in_hist_di <= TARGET_INTERP_MAX_DEPTH_IN_HIST_DI)
                combined_mask = current_sensor_dist_mask & hist_di_depth_mask
                candidate_debug_indices = original_fp_indices_projected[combined_mask].tolist()
                print(f"  Found {len(candidate_debug_indices)} FPs within current sensor range [{MIN_DIST_FOR_FP_SELECTION}m, {MAX_DIST_FOR_FP_SELECTION}m] AND projected depth in HistDI within [{TARGET_INTERP_MIN_DEPTH_IN_HIST_DI}m, {TARGET_INTERP_MAX_DEPTH_IN_HIST_DI}m].")
            else: # Just filter by current sensor distance
                candidate_debug_indices = original_fp_indices_projected[current_sensor_dist_mask].tolist()
                print(f"  Found {len(candidate_debug_indices)} FPs within current sensor range [{MIN_DIST_FOR_FP_SELECTION}m, {MAX_DIST_FOR_FP_SELECTION}m]. Interpolation will depend on config.")

    if candidate_debug_indices:
        PT_IDX_FOR_INTERP_DEBUG = np.random.choice(candidate_debug_indices)
        print(f"Randomly selected FP P{PT_IDX_FOR_INTERP_DEBUG} for MCC/Interpolation Debug from the filtered list.")
    elif misclassified_summary_interp and misclassified_summary_interp.get('fp_dynamic'):
        PT_IDX_FOR_INTERP_DEBUG = np.random.choice(misclassified_summary_interp['fp_dynamic'])
        print(f"No FPs in specified distance/projection criteria. Randomly selected FP P{PT_IDX_FOR_INTERP_DEBUG} from all FPs.")
    else:
        num_pts_in_di = len(processed_target_di_interp.original_points_global_coords)
        PT_IDX_FOR_INTERP_DEBUG = min(15000, num_pts_in_di -1) if num_pts_in_di > 0 else 0
        if num_pts_in_di == 0: PT_IDX_FOR_INTERP_DEBUG = None
        print(f"No suitable FPs found. Using fallback pt_idx_to_debug: P{PT_IDX_FOR_INTERP_DEBUG}")
        
    # 4. Prepare point_global and historical_di for the selected PT_IDX_FOR_INTERP_DEBUG
    # (This part of your Cell 4 remains largely the same)
    if PT_IDX_FOR_INTERP_DEBUG is not None and \
       0 <= PT_IDX_FOR_INTERP_DEBUG < len(processed_target_di_interp.original_points_global_coords):
        
        point_global_for_interp_debug = processed_target_di_interp.original_points_global_coords[PT_IDX_FOR_INTERP_DEBUG, :3]
        print(f"  Debugging Point P{PT_IDX_FOR_INTERP_DEBUG} Global Coords: {np.round(point_global_for_interp_debug, 3).tolist()}")

        actual_hist_di_lib_idx = idx_of_processed_target_di_interp + RELATIVE_HIST_DI_IDX_FOR_CHECK
        if 0 <= actual_hist_di_lib_idx < len(detector_interp.depth_image_library.get_all_images()):
            historical_di_for_interp_check = detector_interp.depth_image_library.get_image_by_index(actual_hist_di_lib_idx)
            if historical_di_for_interp_check:
                print(f"  Selected Historical DI for check: Index {actual_hist_di_lib_idx} (TS: {historical_di_for_interp_check.timestamp:.2f})")
                _, sph_coords_of_pt_in_hist_di, pixel_indices_of_pt_in_hist_di = \
                    historical_di_for_interp_check.project_point_to_pixel_indices(point_global_for_interp_debug)
                if sph_coords_of_pt_in_hist_di is not None:
                    print(f"    P{PT_IDX_FOR_INTERP_DEBUG} projected into Hist DI: Spherical(phi,th,d)={np.round(sph_coords_of_pt_in_hist_di,3)}, Pixel={pixel_indices_of_pt_in_hist_di}")
                else:
                    print(f"    P{PT_IDX_FOR_INTERP_DEBUG} did not project into the selected historical DI's FoV.")
            else:
                print(f"ERROR: Could not retrieve historical DI at library index {actual_hist_di_lib_idx}.")
                historical_di_for_interp_check = None 
        else:
            print(f"ERROR: Calculated historical DI library index {actual_hist_di_lib_idx} is out of bounds.")
            historical_di_for_interp_check = None 
    elif PT_IDX_FOR_INTERP_DEBUG is None:
        print("No point selected for debug (e.g. target DI was empty).")

else:
    print("Prerequisites for selecting point/historical DI for debug not met (initial checks).")
    
print("\nCell 4: Point and Historical DI Selection for Interpolation Debug - OK")

In [ ]:
# %% Cell 5: Execute is_map_consistent (with Interpolation) and Print Debug Info

if detector_interp and point_global_for_interp_debug is not None and processed_target_di_interp:
    print(f"\n--- Calling is_map_consistent for P{PT_IDX_FOR_INTERP_DEBUG} ---")
    
    # Ensure the logger for map_consistency and interpolation_utils is DEBUG
    logging.getLogger('src.core.m_detector.map_consistency').setLevel(logging.DEBUG)
    logging.getLogger('src.core.m_detector.interpolation_utils').setLevel(logging.DEBUG)

    is_consistent, mcc_debug_output = detector_interp.is_map_consistent(
        point_global=point_global_for_interp_debug,
        current_timestamp=processed_target_di_interp.timestamp, # Timestamp of the DI containing the point
        check_direction='past',
        return_debug_info=True
    )
    
    print(f"\n  Overall Map Consistency Result for P{PT_IDX_FOR_INTERP_DEBUG}: {is_consistent}")
    
    if mcc_debug_output:
        print("\n  Detailed Debug Output from is_map_consistent:")
        # Pretty print the debug dictionary
        # For very nested data, you might want a more sophisticated print
        for key, value in mcc_debug_output.items():
            if key == 'relevant_dis_details' and isinstance(value, list):
                print(f"    {key}: (Details for {len(value)} DIs)")
                for i, di_detail in enumerate(value):
                    print(f"      DI Detail {i} (TS: {di_detail.get('di_timestamp', 'N/A'):.2f}):")
                    for sub_key, sub_val in di_detail.items():
                        if sub_key == 'static_points_in_pixel' and isinstance(sub_val, list):
                            print(f"        {sub_key}: ({len(sub_val)} static points checked in pixel)")
                            # Optionally print details of a few static points
                            # for static_pt_idx, static_pt_detail in enumerate(sub_val[:2]): # Print first 2
                            #    print(f"          StaticPt {static_pt_idx}: {static_pt_detail}")
                        else:
                            print(f"        {sub_key}: {sub_val}")
            else:
                print(f"    {key}: {value}")
else:
    print("Cannot execute is_map_consistent: Prerequisites not met.")

print("\nCell 5: is_map_consistent Execution and Debug - OK")

In [ ]:
# %% Cell 6: K3D Visualization for Interpolation Debug

if RUN_K3D_FOR_INTERP_DEBUG and \
   'point_global_for_interp_debug' in locals() and point_global_for_interp_debug is not None and \
   'historical_di_for_interp_check' in locals() and historical_di_for_interp_check is not None and \
   'sph_coords_of_pt_in_hist_di' in locals() and \
   'config_accessor' in locals() and config_accessor is not None:

    print("\n--- Preparing K3D for Interpolation Debug ---")
    k3d_cfg_interp = config_accessor.get_k3d_plot_params()
    k3d_points_interp_viz: Dict[str, Tuple[np.ndarray, int, float]] = {}

    # 1. The point being debugged
    k3d_points_interp_viz['DebugPoint_P'] = (
        point_global_for_interp_debug.reshape(1,3),
        k3d_cfg_interp.get('poi_color_hex', 0xFFD700), # Gold
        k3d_cfg_interp.get('poi_k3d_point_size', 0.3)
    )

    # 2. All points from the historical DI used for the check
    hist_di_pts_global = historical_di_for_interp_check.get_original_points_global()
    if hist_di_pts_global is not None:
        k3d_points_interp_viz['HistoricalDI_AllPts'] = (
            hist_di_pts_global,
            k3d_cfg_interp.get('hist_di_1_color_hex', 0xADD8E6), # LightBlue
            k3d_cfg_interp.get('hist_di_1_point_size', 0.04)
        )
    
    # 3. Highlight static neighbors in the historical DI that were considered for interpolation
    # This requires access to the neighbors list from interpolate_surface_depth_at_angle
    # For now, let's re-fetch them if the function doesn't return them (or modify it to do so)
    
    # Re-fetch static neighbors (simplified for viz, ideally returned by interpolate_surface_depth_at_angle)
    static_neighbor_coords_for_viz = []
    if sph_coords_of_pt_in_hist_di is not None: # Check if projection was successful
        target_phi_viz = sph_coords_of_pt_in_hist_di[0]
        target_theta_viz = sph_coords_of_pt_in_hist_di[1]
        
        mc_params_interp = config_accessor.get_map_consistency_params()
        # Convert static_labels_for_map_check to values
        static_labels_values_viz = [label.value for label in detector_interp.static_labels_for_map_check if isinstance(label, OcclusionResult)] + \
                                   [label for label in detector_interp.static_labels_for_map_check if isinstance(label, str)]


        if historical_di_for_interp_check.local_sph_coords_for_points is not None and \
           historical_di_for_interp_check.mdet_labels_for_points is not None:
            
            all_sph_hist_viz = historical_di_for_interp_check.local_sph_coords_for_points
            all_labels_hist_viz = historical_di_for_interp_check.mdet_labels_for_points
            
            angular_mask_phi_viz = np.abs(all_sph_hist_viz[:, 0] - target_phi_viz) <= detector_interp.epsilon_phi_map_rad
            angular_mask_theta_viz = np.abs(all_sph_hist_viz[:, 1] - target_theta_viz) <= detector_interp.epsilon_theta_map_rad
            static_mask_viz = np.isin(all_labels_hist_viz, static_labels_values_viz)
            
            valid_neighbor_indices_viz = np.where(angular_mask_phi_viz & angular_mask_theta_viz & static_mask_viz)[0]
            
            if valid_neighbor_indices_viz.size > 0:
                # Sort by angular distance and take top N
                neighbor_data_tuples_viz = []
                for idx_viz in valid_neighbor_indices_viz:
                    neighbor_data_tuples_viz.append(
                        (all_sph_hist_viz[idx_viz, 0], all_sph_hist_viz[idx_viz, 1], idx_viz)
                    )
                neighbor_data_tuples_viz.sort(key=lambda p_data: abs(target_phi_viz - p_data[0]) + abs(target_theta_viz - p_data[1]))
                
                num_to_take = min(len(neighbor_data_tuples_viz), mc_params_interp.get('interpolation_max_neighbors_to_consider', 20))
                selected_neighbor_original_indices = [data[2] for data in neighbor_data_tuples_viz[:num_to_take]]
                
                if selected_neighbor_original_indices and historical_di_for_interp_check.original_points_global_coords is not None:
                    static_neighbor_coords_for_viz = historical_di_for_interp_check.original_points_global_coords[selected_neighbor_original_indices]
                    k3d_points_interp_viz['StaticNeighbors_Used'] = (
                        static_neighbor_coords_for_viz,
                        k3d_cfg_interp.get('static_neighbor_color_hex', 0xFF6347), # Tomato
                        k3d_cfg_interp.get('point_size', 0.05) * 1.5 # Make them slightly larger
                    )
                    print(f"  Visualizing {len(selected_neighbor_original_indices)} static neighbors used for interpolation.")

    # 4. If interpolation was successful (from mcc_debug_output), plot the interpolated point
    # This requires mcc_debug_output to be structured to provide this.
    # For now, we'll skip plotting the exact interpolated point unless interpolate_surface_depth_at_angle returns it.
    # If you modify interpolate_surface_depth_at_angle to return the 3D coord of the interpolated surface point,
    # you can add it here.

    plot_title_interp_k3d = f"MCC Interp Debug: P{PT_IDX_FOR_INTERP_DEBUG} vs Hist DI (TS {historical_di_for_interp_check.timestamp:.2f})"
    
    # Set camera to focus on the debug point
    cam_target = point_global_for_interp_debug
    cam_pos = [cam_target[0] + 10, cam_target[1] - 10, cam_target[2] + 5] # Example offset
    initial_camera_interp_k3d = [
        cam_pos[0], cam_pos[1], cam_pos[2],
        cam_target[0], cam_target[1], cam_target[2],
        0,0,1
    ]

    k3d_plot_interp_debug = create_k3d_plot_with_points(
        points_dict=k3d_points_interp_viz,
        plot_title=plot_title_interp_k3d,
        background_color=k3d_cfg_interp.get('plot_background_color_hex', 0xDDDDDD),
        camera_auto_fit=False, # We are setting camera manually
        initial_camera_settings=initial_camera_interp_k3d
    )
    if k3d_plot_interp_debug:
        display(k3d_plot_interp_debug)
    else:
        print("Failed to create K3D plot for interpolation debug.")

elif not RUN_K3D_FOR_INTERP_DEBUG:
    print("K3D plotting for interpolation debug is disabled.")
else:
    print("Skipping K3D for Interpolation Debug: Prerequisites not met.")

print("\nCell 6: K3D Interpolation Debug Visualization - OK")